## 1. Setup & Dependencies

In [1]:
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.model_selection import GroupKFold
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
os.makedirs('/kaggle/working/', exist_ok=True)
print("Environment ready.")

Environment ready.


## 2. Data Loading & Harmonization

Three global AMR surveillance databases are loaded, standardized, and merged into a single unified dataset.

**Resistance encoding:**
- SIR: R=1, S=0, I=discarded (clinically ambiguous)
- MIC: binary via simplified CLSI/EUCAST breakpoints
- Unique isolate IDs: `{SOURCE}_{original_id}_{year}` to prevent cross-database collisions


In [2]:
ATLAS_PATH    = '/kaggle/input/datasets/uaisamangeldi/atlas-csv/atlas.csv'
SIDERO_PATH   = '/kaggle/input/datasets/uaisamangeldi/ddddddd/sidero_wt.xlsx'
KEYSTONE_PATH = '/kaggle/input/datasets/uaisamangeldi/ddddddd/keystone.xlsx'
COMBINED_OUTPUT = '/kaggle/working/combined_dataset.csv'

def parse_sir_to_binary(sir_str):
    """SIR → binary. Intermediate (I) discarded as clinically ambiguous."""
    if pd.isna(sir_str):
        return np.nan
    s = str(sir_str).strip().upper()
    if s in ['R', 'RESISTANT']:   return 1
    if s in ['S', 'SUSCEPTIBLE']: return 0
    return np.nan

def clean_mic_value(val):
    """Normalize MIC string to float."""
    if pd.isna(val): return np.nan
    val = str(val).strip().replace(',', '.')
    for sym in ['<=', '≤', '>=', '≥']:
        val = val.replace(sym, '')
    if '>' in val:
        try: return float(val.replace('>', '').strip()) + 0.0001
        except: return np.nan
    try: return float(val.strip())
    except: return np.nan

def parse_mic_to_binary(mic_val, antibiotic):
    """MIC → binary using simplified CLSI/EUCAST breakpoints."""
    mic = clean_mic_value(mic_val)
    if pd.isna(mic): return np.nan
    thresholds = {
        'Meropenem': 4, 'Imipenem': 4, 'Ceftazidime': 8,
        'Ciprofloxacin': 1, 'Colistin': 4, 'Amikacin': 16,
        'Gentamicin': 8, 'Cefepime': 8,
    }
    return 1 if mic > thresholds.get(antibiotic, 8) else 0

def assign_unique_isolate_id(df, data_source):
    """Composite isolate ID: {source}_{original_id}_{year} — prevents cross-database collisions."""
    if data_source == 'ATLAS' and 'Isolate Id' in df.columns:
        df['original_id'] = df['Isolate Id'].astype(str).str.strip()
    elif data_source == 'KEYSTONE' and 'Collection Number' in df.columns:
        df['original_id'] = df['Collection Number'].astype(str).str.strip()
    else:
        df['original_id'] = df.index.astype(str)
    df['isolate_id'] = data_source + '_' + df['original_id'] + '_' + df['year'].fillna(0).astype(int).astype(str)
    return df

def process_dataset(path, data_source, chunksize=250000):
    print(f"Processing {data_source}...")
    is_csv = path.endswith('.csv')
    reader = pd.read_csv if is_csv else pd.read_excel
    chunks = reader(path, chunksize=chunksize, low_memory=False) if is_csv else [reader(path)]
    processed = []

    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}: {len(chunk)} rows")
        rename = {
            'Isolate Id': 'original_id_temp', 'Species': 'organism',
            'Country': 'country', 'Year': 'year',
            'Organism Name': 'organism', 'Year Collected': 'year',
            'Organism': 'organism', 'Study Year': 'year',
            'Collection Number': 'original_id_temp',
        }
        chunk = chunk.rename(columns=rename)
        for col in ['country', 'year', 'organism']:
            if col not in chunk.columns:
                chunk[col] = 'Unknown' if col != 'year' else np.nan
        chunk['country']  = chunk['country'].astype(str).str.strip()
        chunk['organism'] = chunk['organism'].astype(str).str.strip()
        chunk['year']     = pd.to_numeric(chunk['year'], errors='coerce').astype('Int16')
        chunk = assign_unique_isolate_id(chunk, data_source)

        sir_cols = [c for c in chunk.columns if c.endswith('_I')]
        mic_cols = [c for c in chunk.columns if c not in
                    sir_cols + ['isolate_id', 'country', 'year', 'organism', 'original_id', 'original_id_temp']]
        value_vars   = sir_cols if sir_cols else mic_cols
        melt_varname = 'antibiotic_raw' if sir_cols else 'antibiotic'
        if not value_vars: continue

        long = pd.melt(chunk, id_vars=['isolate_id', 'country', 'year', 'organism'],
                       value_vars=value_vars, var_name=melt_varname, value_name='test_result')

        if sir_cols:
            long['antibiotic'] = long['antibiotic_raw'].str.replace('_I$', '', regex=True).str.strip()
            long['resistance']  = long['test_result'].apply(parse_sir_to_binary)
        else:
            long['antibiotic'] = long['antibiotic'].str.strip()
            long['resistance']  = long.apply(lambda r: parse_mic_to_binary(r['test_result'], r['antibiotic']), axis=1)

        long = long.dropna(subset=['resistance'])
        long['resistance']   = long['resistance'].astype('int8')
        long['data_source']  = data_source
        processed.append(long)

    result = pd.concat(processed, ignore_index=True)
    print(f"  → {len(result):,} rows")
    return result

atlas    = process_dataset(ATLAS_PATH,    'ATLAS')
sidero   = process_dataset(SIDERO_PATH,   'SIDERO-WT')
keystone = process_dataset(KEYSTONE_PATH, 'KEYSTONE')

combined = pd.concat([atlas, sidero, keystone], ignore_index=True)
combined.to_csv(COMBINED_OUTPUT, index=False)
print(f"\nCombined: {len(combined):,} rows | {combined['isolate_id'].nunique():,} unique isolates")
print(f"Class distribution:\n{combined['resistance'].value_counts(normalize=True).round(4)}")

Processing ATLAS...
  Chunk 1: 250000 rows
  Chunk 2: 250000 rows
  Chunk 3: 250000 rows
  Chunk 4: 216805 rows
  → 10,717,988 rows
Processing SIDERO-WT...
  Chunk 1: 47615 rows
  → 435,633 rows
Processing KEYSTONE...
  Chunk 1: 90208 rows
  → 1,296,305 rows

Combined: 12,449,926 rows | 1,100,391 unique isolates
Class distribution:
resistance
0    0.8036
1    0.1964
Name: proportion, dtype: float64


In [3]:
# =============================================================================
# 4. Global Macroeconomic + Per-class Antibiotic Consumption + NEW FEATURES
# =============================================================================
import os
import gc
import pandas as pd
import numpy as np

# ── Original Paths ───────────────────────────────────────────────────────────
POP_DENSITY_PATH = '/kaggle/input/datasets/uaisamangeldi/macrob/API_EN.POP.DNST_DS2_en_csv_v2_275/API_EN.POP.DNST_DS2_en_csv_v2_275.csv'
GDP_PCAP_PATH    = '/kaggle/input/datasets/uaisamangeldi/macrob/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_31/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_31.csv'
HEALTH_GDP_PATH  = '/kaggle/input/datasets/uaisamangeldi/macrob/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_938/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_938.csv'
CONSUMPTION_PATH = '/kaggle/input/datasets/uaisamangeldi/consumption/antibiotic-consumption-rate.csv'
GLOBAL_DID_CLASS_PATH = '/kaggle/input/datasets/uaisamangeldi/globalddd/antibiotic_consumption_clean.csv'

# ── NEW FEATURE PATHS ───────────────────────────────────────────────────────
SANITATION_PATH    = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874.csv'
WATER_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137.csv'
HOSPITAL_BEDS_PATH = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206.csv'
TEMPERATURE_PATH   = '/kaggle/input/datasets/uaisamangeldi/highfeature/average-monthly-surface-temperature/average-monthly-surface-temperature.csv'
AWARE_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/data.csv'

# ── Helper: downcast numerics ────────────────────────────────────────────────
def downcast(df):
    for col in df.select_dtypes(include='float').columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include='integer').columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

# ── Helper: WB long format ───────────────────────────────────────────────────
def wb_long_format(path, value_col_name):
    if not os.path.exists(path):
        print(f" [MISSING] {path}")
        return pd.DataFrame()
    df = pd.read_csv(path, skiprows=4, low_memory=False)
    df.columns = df.columns.str.strip()
    if 'Country Name' not in df.columns:
        print(f" [ERROR] 'Country Name' missing in {path}")
        return pd.DataFrame()
    year_cols = [c for c in df.columns if c.strip().isdigit() and int(c.strip()) >= 2000]
    df_long = pd.melt(df, id_vars=['Country Name'], value_vars=year_cols,
                      var_name='year', value_name=value_col_name)
    del df; gc.collect()
    df_long['year'] = df_long['year'].astype(int)
    df_long = df_long.rename(columns={'Country Name': 'country'})
    df_long[value_col_name] = pd.to_numeric(df_long[value_col_name], errors='coerce')
    result = df_long[['country', 'year', value_col_name]].dropna(subset=[value_col_name])
    del df_long; gc.collect()
    result = downcast(result)
    result['country'] = result['country'].astype('category')
    print(f" {value_col_name}: {len(result):,} rows, {result['country'].nunique()} countries")
    return result

# ── Country Mapping ─────────────────────────────────────────────────────────
COUNTRY_MAP = {
    'United States': 'USA', 'United States of America': 'USA',
    'Russian Federation': 'Russia', 'Korea, Rep.': 'South Korea',
}

# ── Load full_df with categorical strings ────────────────────────────────────
full_df = pd.read_csv("/kaggle/working/combined_dataset.csv")
full_df['country'] = full_df['country'].astype('category')

# ── Core World Bank Indicators ──────────────────────────────────────────────
print("Loading World Bank indicators...")
df_gdp  = wb_long_format(GDP_PCAP_PATH,    'gdp_per_capita_usd')
df_pop  = wb_long_format(POP_DENSITY_PATH, 'population_density_per_sq_km')
df_health = wb_long_format(HEALTH_GDP_PATH, 'health_expenditure_pct_gdp')

macro_df = df_gdp.copy()
del df_gdp; gc.collect()

for df_temp in [df_pop, df_health]:
    macro_df = macro_df.merge(df_temp, on=['country', 'year'], how='outer')
    del df_temp; gc.collect()

del df_pop, df_health; gc.collect()

# ── New World Bank Indicators ───────────────────────────────────────────────
print("\nLoading new World Bank indicators...")
for path, name in [
    (SANITATION_PATH,    'sanitation_access_pct'),
    (WATER_PATH,         'water_access_pct'),
    (HOSPITAL_BEDS_PATH, 'hospital_beds_per_1000'),
]:
    df_temp = wb_long_format(path, name)
    if not df_temp.empty:
        macro_df = macro_df.merge(df_temp, on=['country', 'year'], how='left')
    del df_temp; gc.collect()

# ── Temperature (Parse year from 'Day' column) ──────────────────────────────
print("\nLoading & aggregating temperature data...")
try:
    df_temp = pd.read_csv(TEMPERATURE_PATH, usecols=['Entity', 'Day', 'Monthly average'])
    print(f"Raw temperature shape: {df_temp.shape}")

    df_temp['year'] = pd.to_datetime(df_temp['Day'], errors='coerce').dt.year
    df_temp = df_temp.drop(columns=['Day'])  # free Day column immediately

    df_temp_yearly = df_temp.groupby(['Entity', 'year'], as_index=False)['Monthly average'].mean()
    del df_temp; gc.collect()

    df_temp_yearly = df_temp_yearly.rename(columns={
        'Entity': 'country',
        'Monthly average': 'mean_temp_celsius'
    })
    df_temp_yearly['country'] = df_temp_yearly['country'].replace(COUNTRY_MAP)
    df_temp_yearly = df_temp_yearly[['country', 'year', 'mean_temp_celsius']].dropna()
    df_temp_yearly['year'] = df_temp_yearly['year'].astype(int)
    df_temp_yearly = downcast(df_temp_yearly)

    macro_df = macro_df.merge(df_temp_yearly, on=['country', 'year'], how='left')
    del df_temp_yearly; gc.collect()
    print(f" mean_temp_celsius: merged OK")
except Exception as e:
    print(f" Temperature FAILED: {e}")
    macro_df['mean_temp_celsius'] = np.nan

# ── AWaRe Access % ──────────────────────────────────────────────────────────
print("\nLoading AWaRe Access % data...")
try:
    df_aware = pd.read_csv(AWARE_PATH, usecols=['Location', 'Period', 'FactValueNumeric'])
    df_aware = df_aware.rename(columns={
        'Location':         'country',
        'Period':           'year',
        'FactValueNumeric': 'aware_access_pct',
    })
    df_aware['country'] = df_aware['country'].replace(COUNTRY_MAP)
    df_aware = df_aware[['country', 'year', 'aware_access_pct']].dropna()
    df_aware['year'] = df_aware['year'].astype(int)
    df_aware = downcast(df_aware)

    macro_df = macro_df.merge(df_aware, on=['country', 'year'], how='left')
    del df_aware; gc.collect()
    print(f" aware_access_pct: merged OK")
except Exception as e:
    print(f" AWaRe FAILED: {e}")
    macro_df['aware_access_pct'] = np.nan

# ── Downcast macro_df before merging into full_df ───────────────────────────
macro_df = downcast(macro_df)
gc.collect()

# ── Cleanup & Antibiotic Class Mapping ──────────────────────────────────────
full_df['country'] = full_df['country'].astype(str).replace(COUNTRY_MAP).astype('category')
full_df = full_df[~full_df['antibiotic'].str.strip().isin(['Age', 'Date Collected', 'age', 'date collected'])]

ANTIBIOTIC_TO_CLASS_MAP = {
    'Meropenem': 'Carbapenems',           'Imipenem': 'Carbapenems',
    'Doripenem': 'Carbapenems',           'Ertapenem': 'Carbapenems',
    'Ciprofloxacin': 'Fluoroquinolones',  'Levofloxacin': 'Fluoroquinolones',
    'Moxifloxacin': 'Fluoroquinolones',
    'Ceftazidime': 'Cephalosporins',      'Cefepime': 'Cephalosporins',
    'Ceftriaxone': 'Cephalosporins',
    'Amikacin': 'Aminoglycosides',        'Gentamicin': 'Aminoglycosides',
    'Ampicillin': 'Penicillins',          'Amoxycillin clavulanate': 'Penicillins',
    'Piperacillin': 'Penicillins',
    'Azithromycin': 'Macrolides',         'Erythromycin': 'Macrolides',
    'Vancomycin': 'Glycopeptides',        'Linezolid': 'Oxazolidinones',
    'Colistin': 'Polymyxins',             'Tigecycline': 'Tetracyclines',
    'Trimethoprim sulfa': 'Sulfonamides',
}

full_df['antibiotic_class'] = full_df['antibiotic'].map(ANTIBIOTIC_TO_CLASS_MAP).astype('category')
full_df['antibiotic']       = full_df['antibiotic'].astype('category')

# ── Drop Raw Columns (Keep antibiotic_class) ────────────────────────────────
drop_cols = [
    'gdp_per_capita_usd', 'population_density_per_sq_km', 'health_expenditure_pct_gdp',
    'consumption_ddd_per_1000_day', 'consumption_ddd_total', 'consumption_did_per_class',
    'log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class',
    'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000',
    'mean_temp_celsius', 'aware_access_pct',
]
full_df = full_df.drop(columns=[c for c in drop_cols if c in full_df.columns])

full_df = full_df.merge(macro_df, on=['country', 'year'], how='left')
del macro_df; gc.collect()

# ── Total Consumption ────────────────────────────────────────────────────────
print("\nLoading total consumption...")
try:
    df_cons = pd.read_csv(CONSUMPTION_PATH, usecols=[
        'Entity', 'Year',
        'Defined daily doses of antibiotics and antituberculosis drugs used per 1,000 inhabitants per day'
    ])
    df_cons = df_cons.rename(columns={
        'Entity': 'country',
        'Year':   'year',
        'Defined daily doses of antibiotics and antituberculosis drugs used per 1,000 inhabitants per day': 'consumption_ddd_total'
    })
    df_cons['country'] = df_cons['country'].replace({'Russian Federation': 'Russia', 'United States': 'USA'})
    df_cons = downcast(df_cons)

    full_df = full_df.merge(df_cons, on=['country', 'year'], how='left')
    del df_cons; gc.collect()
    print(" → Total consumption OK")
except Exception as e:
    print(f" → Total consumption FAILED: {e}")
    full_df['consumption_ddd_total'] = np.nan

# ── Per-class DDD ────────────────────────────────────────────────────────────
print("\nLoading per-class DDD...")
did_raw = pd.read_csv(GLOBAL_DID_CLASS_PATH,
                      usecols=['country', 'year', 'antibiotic_class', 'consumption_DID'])
DID_COUNTRY_MAP = {
    'Us': 'USA', 'Uk': 'United Kingdom', 'Uae': 'United Arab Emirates',
    'Korea': 'South Korea', 'Russian Federation': 'Russia'
}
did_raw['country'] = did_raw['country'].replace(DID_COUNTRY_MAP)
did_raw = did_raw.rename(columns={'consumption_DID': 'consumption_did_per_class'})
did_raw = downcast(did_raw)

full_df = full_df.merge(did_raw, on=['country', 'year', 'antibiotic_class'], how='left')
del did_raw; gc.collect()

# ── Imputation ──────────────────────────────────────────────────────────────
fill_cols = [
    'gdp_per_capita_usd', 'population_density_per_sq_km', 'health_expenditure_pct_gdp',
    'consumption_ddd_total', 'consumption_did_per_class',
    'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000',
    'mean_temp_celsius', 'aware_access_pct'
]

# Convert categoricals back to string for groupby compatibility
for col in ['country', 'antibiotic', 'antibiotic_class']:
    if col in full_df.columns and full_df[col].dtype.name == 'category':
        full_df[col] = full_df[col].astype(str)

for col in fill_cols:
    if col in full_df.columns:
        full_df[col] = full_df.groupby('country')[col].transform(lambda x: x.ffill().bfill())
        full_df[col] = full_df[col].fillna(full_df[col].mean())

# ── Log Transforms ──────────────────────────────────────────────────────────
full_df['log_gdp_per_capita']            = np.log1p(full_df['gdp_per_capita_usd'].clip(lower=0))
full_df['log_consumption_ddd']           = np.log1p(full_df['consumption_ddd_total'].clip(lower=0))
full_df['log_consumption_did_per_class'] = np.log1p(full_df['consumption_did_per_class'].clip(lower=0))

full_df = downcast(full_df)
gc.collect()

# ── Save ────────────────────────────────────────────────────────────────────
os.makedirs('/kaggle/working', exist_ok=True)
full_df.to_csv('/kaggle/working/full_amr_with_macro.csv', index=False)

print(f"\n✅ Successfully saved full_amr_with_macro.csv | Shape: {full_df.shape}")
print("\nCoverage (non-null %):")
print((full_df[fill_cols + ['log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class']]
       .notna().mean() * 100).round(1))

Loading World Bank indicators...
 gdp_per_capita_usd: 6,433 rows, 262 countries
 population_density_per_sq_km: 6,217 rows, 264 countries
 health_expenditure_pct_gdp: 5,716 rows, 241 countries

Loading new World Bank indicators...
 sanitation_access_pct: 4,319 rows, 176 countries
 water_access_pct: 6,398 rows, 262 countries
 hospital_beds_per_1000: 3,657 rows, 226 countries

Loading & aggregating temperature data...
Raw temperature shape: (220242, 3)
 mean_temp_celsius: merged OK

Loading AWaRe Access % data...
 aware_access_pct: merged OK

Loading total consumption...
 → Total consumption OK

Loading per-class DDD...

✅ Successfully saved full_amr_with_macro.csv | Shape: (20446868, 23)

Coverage (non-null %):
gdp_per_capita_usd               100.0
population_density_per_sq_km     100.0
health_expenditure_pct_gdp       100.0
consumption_ddd_total            100.0
consumption_did_per_class        100.0
sanitation_access_pct            100.0
water_access_pct                 100.0
hospital

In [3]:
full_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")

In [16]:
df_aware = df_aware.rename(columns={
    'Location': 'country',
    'Period': 'year',
    'FactValueNumeric': 'aware_access_pct',
})
# Drop 'Value' column before selecting to avoid duplicate
df_aware = df_aware[['country', 'year', 'aware_access_pct']].dropna()
df_aware.head()

,country,year,aware_access_pct,aware_access_pct
0,Armenia,2023,0.001,0.0
1,Austria,2023,0.000,0.0
2,Belgium,2023,0.000,0.0
3,Benin,2023,0.000,0.0
4,Bhutan,2023,0.000,0.0


In [4]:
full_df = pd.read_csv('/kaggle/working/full_amr_with_macro.csv', low_memory=False)
full_df_sorted = full_df.sort_values('year').reset_index(drop=True)

TRAIN_CUTOFF, VAL_CUTOFF = 2018, 2020

train = full_df_sorted[full_df_sorted['year'] <= TRAIN_CUTOFF]
val   = full_df_sorted[(full_df_sorted['year'] > TRAIN_CUTOFF) & (full_df_sorted['year'] <= VAL_CUTOFF)]
test  = full_df_sorted[full_df_sorted['year'] > VAL_CUTOFF]

print(f"Train : {len(train):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})")
print(f"Test  : {len(test):,} rows (> {VAL_CUTOFF})")

train_ids, val_ids, test_ids = set(train['isolate_id']), set(val['isolate_id']), set(test['isolate_id'])
print(f"\nTrain-Val overlap  : {len(train_ids & val_ids):,}")
print(f"Train-Test overlap : {len(train_ids & test_ids):,}")
print(f"Val-Test overlap   : {len(val_ids & test_ids):,}")
print("\nLeakage audit PASSED." if len(train_ids & test_ids) == 0 else "\nWARNING: Leakage detected.")

Train : 11,207,215 rows (≤ 2018)
Val   : 3,607,997 rows (2019–2020)
Test  : 5,631,656 rows (> 2020)

Train-Val overlap  : 0
Train-Test overlap : 0
Val-Test overlap   : 0

Leakage audit PASSED.


## 5. Feature Set Definition

Country name is deliberately excluded to enable zero-shot generalization.
The model characterizes each country purely through its macroeconomic profile.

Feature set matches the final validated pipeline from the original notebook.


In [4]:
# =============================================================================
# Feature Set Definition (Cell 18)
# =============================================================================

CAT_FEATURES = [
    'organism', 
    'antibiotic', 
    'antibiotic_class'         # important for zero-shot
]

NUM_FEATURES = [
    'year',
    'log_gdp_per_capita',
    'health_expenditure_pct_gdp',
    'population_density_per_sq_km',
    'log_consumption_ddd',
    'log_consumption_did_per_class',
    # ── NEW MACRO FEATURES ─────────────────────────────────────
    'sanitation_access_pct',
    'water_access_pct',
    'hospital_beds_per_1000',
    'mean_temp_celsius',
    'aware_access_pct'
]

FEATURES = CAT_FEATURES + NUM_FEATURES

print("Categorical features :", CAT_FEATURES)
print("Numerical features   :", NUM_FEATURES)
print("Total features       :", len(FEATURES))
print("\n✅ Feature list ready for training.")

Categorical features : ['organism', 'antibiotic', 'antibiotic_class']
Numerical features   : ['year', 'log_gdp_per_capita', 'health_expenditure_pct_gdp', 'population_density_per_sq_km', 'log_consumption_ddd', 'log_consumption_did_per_class', 'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius', 'aware_access_pct']
Total features       : 14

✅ Feature list ready for training.


In [6]:
# =============================================================================
# FULL LOCO + BRIER SCORE ANALYSIS (Compatible with your Feature Set)
# =============================================================================
import gc
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

print("=== LOCO + Brier Score Analysis for Kazakhstan ===\n")

# ----------------------------- YOUR FEATURE SET -----------------------------
CAT_FEATURES = ['organism', 'antibiotic', 'antibiotic_class']
NUM_FEATURES = [
    'year', 'log_gdp_per_capita', 'health_expenditure_pct_gdp',
    'population_density_per_sq_km', 'log_consumption_ddd',
    'log_consumption_did_per_class', 'sanitation_access_pct',
    'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius',
    'aware_access_pct'
]
FEATURES = CAT_FEATURES + NUM_FEATURES

print(f"Using {len(FEATURES)} features ({len(CAT_FEATURES)} categorical)\n")

# ----------------------------- LOAD DATA -----------------------------
full_df = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv', low_memory=False)
kz_final = pd.read_csv('/kaggle/working/kazakhstan_final_prediction_input_2017_2023.csv')

print(f"Global data : {len(full_df):,} rows")
print(f"KZ base     : {len(kz_final):,} rows\n")

# ----------------------------- PREPARE KZ DATA -----------------------------
kz_final['country'] = 'Kazakhstan'      # still useful for tracking
kz_final['data_source'] = 'KZ_DID'

TARGET_ORGANISMS = [
    'Escherichia coli', 'Klebsiella pneumoniae', 'Staphylococcus aureus',
    'Pseudomonas aeruginosa', 'Acinetobacter baumannii', 'Enterococcus faecium'
]

prediction_rows = []
for _, row in kz_final.iterrows():
    for org in TARGET_ORGANISMS:
        new_row = row.to_dict()
        new_row['organism'] = org
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
print(f"KZ prediction rows: {len(kz_df):,}\n")

# ----------------------------- CONFIG -----------------------------
TEMPORAL_CUTOFF = 2020
ES_VAL_YEARS = [2019, 2020]

loco_groups = {
    'post_soviet':      ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia'],
    'structural_peers': ['Qatar', 'Turkey', 'Romania', 'Bulgaria', 'Hungary', 'Poland'],
    'all_except_kz':    [c for c in full_df['country'].unique() if c != 'Kazakhstan']
}

# ----------------------------- SCALE POS WEIGHT -----------------------------
source_df = full_df[~full_df['country'].isin(['Kazakhstan', 'Russia', 'Belarus', 'Ukraine'])].copy()
SCALE_POS_WEIGHT = (source_df['resistance'] == 0).sum() / (source_df['resistance'] == 1).sum()
print(f"SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT:.4f}\n")

# ----------------------------- CATBOOST PARAMS -----------------------------
CATBOOST_PARAMS = {
    'learning_rate': 0.03,
    'depth': 7,
    'l2_leaf_reg': 3,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 200,
    'early_stopping_rounds': 100,
    'iterations': 5000,
    'task_type': 'GPU',
    'scale_pos_weight': SCALE_POS_WEIGHT,
}

# ----------------------------- POOL HELPERS -----------------------------
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def make_kz_pool(df):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    return Pool(data=X, cat_features=CAT_FEATURES)

# ----------------------------- LOCO LOOP -----------------------------
loco_results = {}

for group_name, countries in loco_groups.items():
    print(f"{'='*75}")
    print(f"[{group_name.upper()}] — {len(countries)} countries")
    
    available = [c for c in countries if c in full_df['country'].unique()]
    train_source = full_df[full_df['country'].isin(available)].copy()
    
    es_train_df = train_source[(train_source['year'] <= TEMPORAL_CUTOFF) & 
                              (~train_source['year'].isin(ES_VAL_YEARS))].copy()
    es_val_df = train_source[train_source['year'].isin(ES_VAL_YEARS)].copy()
    
    if len(es_val_df) < 200:
        print("  Skipped — val set too small")
        continue
    
    # Train
    model = CatBoostClassifier(**CATBOOST_PARAMS)
    model.fit(make_pool(es_train_df), eval_set=make_pool(es_val_df))
    best_iter = model.best_iteration_
    
    # Final retrain
    final_train = train_source[train_source['year'] <= TEMPORAL_CUTOFF].copy()
    final_params = {**CATBOOST_PARAMS, 'iterations': best_iter, 
                    'early_stopping_rounds': None, 'verbose': 100}
    model = CatBoostClassifier(**final_params)
    model.fit(make_pool(final_train))
    
    # Predict Kazakhstan
    kz_proba = model.predict_proba(make_kz_pool(kz_df))[:, 1]
    
    # Brier Score on validation set
    val_proba = model.predict_proba(make_pool(es_val_df))[:, 1]
    y_val = es_val_df['resistance'].values
    val_brier = brier_score_loss(y_val, val_proba)
    val_auc = roc_auc_score(y_val, val_proba)
    
    print(f"  Val Brier = {val_brier:.4f} | AUC = {val_auc:.4f}")
    print(f"  KZ Avg Prob = {kz_proba.mean():.4f}\n")
    
    loco_results[group_name] = {'kz_proba': kz_proba, 'val_brier': val_brier, 'val_auc': val_auc}
    
    del model, final_train, es_train_df, es_val_df
    gc.collect()

# ----------------------------- SUMMARY -----------------------------
print("="*75)
print("LOCO BRIER SCORE SUMMARY")
summary = []
for g, r in loco_results.items():
    summary.append({
        'Group': g,
        'Val_Brier': round(r['val_brier'], 4),
        'Val_AUC': round(r['val_auc'], 4),
        'KZ_Mean_Prob': round(r['kz_proba'].mean(), 4)
    })

summary_df = pd.DataFrame(summary).sort_values('Val_Brier')
print(summary_df)

# Ensemble
for g, r in loco_results.items():
    kz_df[f'prob_{g}'] = r['kz_proba']

kz_df['prob_ensemble'] = kz_df[[f'prob_{g}' for g in loco_results]].mean(axis=1)

kz_df.to_csv('/kaggle/working/kz_loco_final.csv', index=False)
summary_df.to_csv('/kaggle/working/loco_brier_summary.csv', index=False)

print("\n✅ LOCO Analysis Completed Successfully!")
print("Best group (lowest Brier):", summary_df.iloc[0]['Group'])

=== LOCO + Brier Score Analysis for Kazakhstan ===

Using 14 features (3 categorical)

Global data : 20,446,868 rows
KZ base     : 183 rows

KZ prediction rows: 1,098

SCALE_POS_WEIGHT : 4.5446

[POST_SOVIET] — 6 countries


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8604288	best: 0.8604288 (0)	total: 165ms	remaining: 13m 43s
200:	test: 0.8949424	best: 0.8949466 (199)	total: 6.75s	remaining: 2m 41s
400:	test: 0.8962906	best: 0.8962906 (400)	total: 13.3s	remaining: 2m 32s
600:	test: 0.8967608	best: 0.8967608 (600)	total: 19.9s	remaining: 2m 25s
800:	test: 0.8969371	best: 0.8969516 (799)	total: 26.4s	remaining: 2m 18s
bestTest = 0.8969516158
bestIteration = 799
Shrink model to first 800 iterations.


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 35.8ms	remaining: 28.5s
100:	total: 3.64s	remaining: 25.1s
200:	total: 7.22s	remaining: 21.5s
300:	total: 10.8s	remaining: 17.8s
400:	total: 14.4s	remaining: 14.3s
500:	total: 17.9s	remaining: 10.6s
600:	total: 21.4s	remaining: 7.04s
700:	total: 25s	remaining: 3.49s
798:	total: 28.5s	remaining: 0us
  Val Brier = 0.1490 | AUC = 0.9226
  KZ Avg Prob = 0.5079

[STRUCTURAL_PEERS] — 6 countries


...

0:	total: 92.3ms	remaining: 49.7s
100:	total: 8.64s	remaining: 37.5s
200:	total: 17.3s	remaining: 29s
300:	total: 25.6s	remaining: 20.2s
400:	total: 34.1s	remaining: 11.7s
500:	total: 42.5s	remaining: 3.22s
538:	total: 45.6s	remaining: 0us
  Val Brier = 0.1516 | AUC = 0.9096
  KZ Avg Prob = 0.5278

[ALL_EXCEPT_KZ] — 85 countries


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8433207	best: 0.8433207 (0)	total: 1.08s	remaining: 1h 30m 24s
200:	test: 0.8800893	best: 0.8800893 (200)	total: 3m 19s	remaining: 1h 19m 31s
400:	test: 0.8834674	best: 0.8834674 (400)	total: 6m 14s	remaining: 1h 11m 34s
600:	test: 0.8846955	best: 0.8846955 (600)	total: 9m 15s	remaining: 1h 7m 43s
800:	test: 0.8855096	best: 0.8855204 (799)	total: 12m 14s	remaining: 1h 4m 9s
1000:	test: 0.8859949	best: 0.8859949 (1000)	total: 15m 16s	remaining: 1h 1m 1s
1200:	test: 0.8865685	best: 0.8865831 (1198)	total: 18m 25s	remaining: 58m 17s
1400:	test: 0.8870449	best: 0.8870876 (1365)	total: 21m 33s	remaining: 55m 22s
1600:	test: 0.8875698	best: 0.8875698 (1599)	total: 24m 46s	remaining: 52m 36s
1800:	test: 0.8876378	best: 0.8876737 (1781)	total: 27m 57s	remaining: 49m 39s
2000:	test: 0.8879754	best: 0.8880077 (1998)	total: 31m 2s	remaining: 46m 31s
2200:	test: 0.8880987	best: 0.8881032 (2199)	total: 34m 16s	remaining: 43m 35s
2400:	test: 0.8882966	best: 0.8882966 (2400)	total: 37m 27s

Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 1.05s	remaining: 53m 5s
100:	total: 2m 16s	remaining: 1h 5m 31s
200:	total: 4m 16s	remaining: 1h 3s
300:	total: 6m 10s	remaining: 55m 46s
400:	total: 8m 3s	remaining: 52m 38s
500:	total: 9m 56s	remaining: 50m
600:	total: 11m 50s	remaining: 47m 39s
700:	total: 13m 43s	remaining: 45m 23s
800:	total: 15m 43s	remaining: 43m 34s
900:	total: 17m 36s	remaining: 41m 24s
1000:	total: 19m 35s	remaining: 39m 30s
1100:	total: 21m 35s	remaining: 37m 38s
1200:	total: 23m 30s	remaining: 35m 36s
1300:	total: 25m 31s	remaining: 33m 43s
1400:	total: 27m 30s	remaining: 31m 47s
1500:	total: 29m 29s	remaining: 29m 50s
1600:	total: 31m 27s	remaining: 27m 52s
1700:	total: 33m 23s	remaining: 25m 53s
1800:	total: 35m 24s	remaining: 23m 58s
1900:	total: 37m 24s	remaining: 22m 1s
2000:	total: 39m 24s	remaining: 20m 4s
2100:	total: 41m 20s	remaining: 18m 4s
2200:	total: 43m 23s	remaining: 16m 8s
2300:	total: 45m 18s	remaining: 14m 9s
2400:	total: 47m 16s	remaining: 12m 11s
2500:	total: 49m 13s	remaining

In [14]:
import pandas as pd
df = pd.read_csv('/kaggle/working/kazakhstan_final_prediction_input_2017_2023.csv')
print(df.shape)
print(df.columns.tolist())
print(df.head(3))
print(df.dtypes)

(183, 15)
['year', 'antibiotic', 'atc_code', 'antibiotic_class', 'did', 'health_expenditure_pct_gdp', 'population_density_per_sq_km', 'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius', 'aware_access_pct', 'log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class']
   year                     antibiotic atc_code antibiotic_class    did  \
0  2017                       amikacin  J01GB06           Access  0.098   
1  2017                    amoxicillin  J01CA04           Access  0.040   
2  2017  amoxicillin / clavulanic acid  J01CR02           Access  0.102   

   health_expenditure_pct_gdp  population_density_per_sq_km  \
0                    3.052445                      6.844997   
1                    3.052445                      6.844997   
2                    3.052445                      6.844997   

   sanitation_access_pct  water_access_pct  hospital_beds_per_1000  \
0              81.909807         95.997018          

In [15]:
# =============================================================================
# TOP 10 HIGHEST RISK ANTIBIOTIC-ORGANISM COMBINATIONS IN KAZAKHSTAN
# =============================================================================
import pandas as pd

# Load the file with LOCO predictions (generated from previous LOCO code)
kz_pred = pd.read_csv('/kaggle/working/kz_loco_final.csv')

print(f"Total prediction rows: {len(kz_pred):,}\n")

# Select relevant columns and sort by ensemble probability (most reliable)
top_risks = kz_pred[[
    'year', 
    'antibiotic', 
    'antibiotic_class',
    'organism', 
    'prob_ensemble'
]].copy()

# Sort by highest predicted resistance probability
top_risks = top_risks.sort_values(by='prob_ensemble', ascending=False)

# Show Top 10
print("🔥 TOP 10 HIGHEST RISK ANTIBIOTIC-ORGANISM COMBINATIONS IN KAZAKHSTAN\n")
print(top_risks.head(10).round(4).to_string(index=False))

# Save full ranked list
top_risks.to_csv('/kaggle/working/kazakhstan_top_risk_combinations.csv', index=False)

print("\n✅ Full ranked list saved as: kazakhstan_top_risk_combinations.csv")

Total prediction rows: 1,098

🔥 TOP 10 HIGHEST RISK ANTIBIOTIC-ORGANISM COMBINATIONS IN KAZAKHSTAN

 year      antibiotic antibiotic_class                organism  prob_ensemble
 2023     nitroxoline     Unclassified Acinetobacter baumannii         0.8927
 2022     nitroxoline     Unclassified Acinetobacter baumannii         0.8917
 2021     nitroxoline     Unclassified Acinetobacter baumannii         0.8890
 2019       linezolid          Reserve Acinetobacter baumannii         0.8882
 2019 fosfomycin (iv)          Reserve Acinetobacter baumannii         0.8882
 2020     nitroxoline     Unclassified Acinetobacter baumannii         0.8881
 2019     nitroxoline     Unclassified Acinetobacter baumannii         0.8850
 2017       linezolid          Reserve Acinetobacter baumannii         0.8648
 2017 fosfomycin (iv)          Reserve Acinetobacter baumannii         0.8648
 2017    moxifloxacin            Watch Acinetobacter baumannii         0.8595

✅ Full ranked list saved as: kazakhstan_t

In [16]:
# Top 5 highest risk per year
print("TOP 5 HIGHEST RISK COMBINATIONS PER YEAR:\n")
for year in sorted(top_risks['year'].unique()):
    yearly = top_risks[top_risks['year'] == year].head(5)
    print(f"Year {year}:")
    print(yearly[['antibiotic', 'organism', 'prob_ensemble']].round(4).to_string(index=False))
    print("-" * 60)

TOP 5 HIGHEST RISK COMBINATIONS PER YEAR:

Year 2017:
     antibiotic                organism  prob_ensemble
      linezolid Acinetobacter baumannii         0.8648
fosfomycin (iv) Acinetobacter baumannii         0.8648
   moxifloxacin Acinetobacter baumannii         0.8595
 clarithromycin Acinetobacter baumannii         0.8595
      ofloxacin Acinetobacter baumannii         0.8595
------------------------------------------------------------
Year 2018:
          antibiotic                organism  prob_ensemble
     fosfomycin (iv) Acinetobacter baumannii         0.8588
           linezolid Acinetobacter baumannii         0.8588
cefpodoxime proxetil Acinetobacter baumannii         0.8456
           meropenem Acinetobacter baumannii         0.8456
     vancomycin (iv) Acinetobacter baumannii         0.8456
------------------------------------------------------------
Year 2019:
     antibiotic                organism  prob_ensemble
      linezolid Acinetobacter baumannii         0.8882
fo

# NEED FIX (BELOW)

In [18]:
# =============================================================================
# LOCO + BRIER + PROPER CALIBRATION (Recommended)
# =============================================================================
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.isotonic import IsotonicRegression

# ... (keep all your previous code until the LOCO loop) ...

loco_results = {}

for group_name, countries in loco_groups.items():
    print(f"[{group_name.upper()}] — Training...")
    
    # ... training code same as before ...
    
    # Predict on Kazakhstan (raw)
    kz_proba_raw = model.predict_proba(make_kz_pool(kz_df))[:, 1]
    
    # Calibrate using Isotonic Regression on validation set
    val_proba_raw = model.predict_proba(make_pool(es_val_df))[:, 1]
    y_val = es_val_df['resistance'].values
    
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(val_proba_raw, y_val)
    
    kz_proba_calibrated = calibrator.predict(kz_proba_raw)
    
    # Metrics
    val_brier = brier_score_loss(y_val, val_proba_raw)
    val_brier_cal = brier_score_loss(y_val, calibrator.predict(val_proba_raw))
    
    print(f"  Raw Brier    = {val_brier:.4f}")
    print(f"  Cal Brier    = {val_brier_cal:.4f} ← Improved")
    print(f"  KZ Avg Prob (calibrated) = {kz_proba_calibrated.mean():.4f}\n")
    
    loco_results[group_name] = {
        'kz_proba': kz_proba_calibrated,   # Use calibrated version
        'val_brier': val_brier_cal,
        'val_auc': roc_auc_score(y_val, val_proba_raw)
    }

# Ensemble using calibrated probabilities
for g, r in loco_results.items():
    kz_df[f'prob_{g}'] = r['kz_proba']

kz_df['prob_ensemble'] = kz_df[[f'prob_{g}' for g in loco_results]].mean(axis=1)

print("✅ Calibration applied using Isotonic Regression")

[POST_SOVIET] — Training...
  Raw Brier    = 0.1490
  Cal Brier    = 0.1029 ← Improved
  KZ Avg Prob (calibrated) = 0.2816

[STRUCTURAL_PEERS] — Training...
  Raw Brier    = 0.1490
  Cal Brier    = 0.1029 ← Improved
  KZ Avg Prob (calibrated) = 0.2816

[ALL_EXCEPT_KZ] — Training...
  Raw Brier    = 0.1490
  Cal Brier    = 0.1029 ← Improved
  KZ Avg Prob (calibrated) = 0.2816

✅ Calibration applied using Isotonic Regression


## 7. Model Training

Three models trained on identical feature sets and temporal splits:
- **Random Forest** — baseline (class_weight='balanced')
- **XGBoost** — GPU-accelerated, histogram method (scale_pos_weight)
- **LightGBM** — GPU-accelerated, leaf-wise growth (scale_pos_weight, early stopping)
- **MLP**
- **CatBoost**
- **Logistic Regression**
Temporal split: Train ≤ 2020 | Validation 2021–2022.
Label encoding fit on training data only — validation labels mapped, unknowns assigned -1.


In [ ]:
!pip install catboost

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler as SklearnScaler
from sklearn.metrics import (
    f1_score, recall_score, precision_score,
    brier_score_loss, roc_auc_score, average_precision_score
)
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from scipy import stats
from scipy.special import expit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pickle
import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020
source_df = pd.read_csv("/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv")
source_sorted = source_df.sort_values('year').reset_index(drop=True)

train_df = source_sorted[source_sorted['year'] <= TRAIN_CUTOFF].copy()
val_df   = source_sorted[(source_sorted['year'] > TRAIN_CUTOFF) &
                          (source_sorted['year'] <= VAL_CUTOFF)].copy()
test_df  = source_sorted[source_sorted['year'] > VAL_CUTOFF].copy()

print(f"Train : {len(train_df):,} rows (≤ {TRAIN_CUTOFF})")
print(f"Val   : {len(val_df):,} rows ({TRAIN_CUTOFF+1}–{VAL_CUTOFF})  [early stopping only]")
print(f"Test  : {len(test_df):,} rows (> {VAL_CUTOFF})  [final reported metrics]")

# ── Label encoding (fit on train only, applied to val + test) ─────────────────
label_encoders = {}

X_train = train_df[FEATURES].copy()
X_val   = val_df[FEATURES].copy()
X_test  = test_df[FEATURES].copy()

y_train = train_df['resistance'].values
y_val   = val_df['resistance'].values
y_test  = test_df['resistance'].values

for col in CAT_FEATURES:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].fillna('unknown').astype(str))
    le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
    X_val[col]  = X_val[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    X_test[col] = X_test[col].fillna('unknown').astype(str).map(
        lambda s, d=le_dict: d.get(s, -1))
    label_encoders[col] = le

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
n_pos_test = int(y_test.sum())
n_neg_test = int(len(y_test) - n_pos_test)
print(f"\nscale_pos_weight : {pos_weight:.2f}")
print(f"Test class dist  : {n_pos_test:,} resistant / {n_neg_test:,} susceptible")

results = {}


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def compute_metrics(y_true, y_prob, threshold=0.5):
    """Full metric suite for one prediction array on test set."""
    y_bin = (y_prob >= threshold).astype(int)
    prevalence = y_true.mean()
    bs         = brier_score_loss(y_true, y_prob)
    bs_ref     = prevalence * (1 - prevalence)   # climatology baseline
    bss        = 1.0 - bs / bs_ref if bs_ref > 0 else np.nan
    return {
        'auc'       : roc_auc_score(y_true, y_prob),
        'auprc'     : average_precision_score(y_true, y_prob),
        'f1'        : f1_score(y_true, y_bin),
        'precision' : precision_score(y_true, y_bin),
        'recall'    : recall_score(y_true, y_bin),
        'brier'     : bs,
        'bss'       : bss,
    }


def bootstrap_ci(y_true, y_prob, metric_fn, n_boot=1000, seed=42, ci=95):
    """
    Percentile bootstrap 95 % CI for a scalar metric.
    metric_fn: callable(y_true, y_prob) → float
    Returns (mean, lower, upper).
    """
    rng = np.random.default_rng(seed)
    vals = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        # skip bootstrap samples with only one class
        if y_true[idx].nunique() < 2 if hasattr(y_true[idx], 'nunique') \
                else len(np.unique(y_true[idx])) < 2:
            continue
        try:
            vals.append(metric_fn(y_true[idx], y_prob[idx]))
        except Exception:
            continue
    alpha = (100 - ci) / 2
    return float(np.mean(vals)), float(np.percentile(vals, alpha)), \
           float(np.percentile(vals, 100 - alpha))


def delong_test(y_true, y_prob_a, y_prob_b):
    """
    DeLong et al. (1988) test for equality of two AUROCs on the same dataset.
    Structural component implementation — exact, not approximation.
    Returns (z_statistic, two_sided_p_value).

    BUG FIX vs original: the original used a single-sample Hanley-McNeil
    variance formula that ignores the covariance between the two AUCs,
    producing incorrect p-values for paired comparisons. This version
    computes the full covariance matrix of the structural components.
    """
    y_true  = np.asarray(y_true)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    n1, n0  = len(pos_idx), len(neg_idx)

    def placement(pos_scores, neg_scores):
        # V10[i] = P(score_pos_i > score_neg) + 0.5*P(score_pos_i == score_neg)
        V10 = np.array([
            np.mean(p > neg_scores) + 0.5 * np.mean(p == neg_scores)
            for p in pos_scores
        ])
        # V01[j] = P(score_neg_j < score_pos) + 0.5*P(score_neg_j == score_pos)
        V01 = np.array([
            np.mean(q < pos_scores) + 0.5 * np.mean(q == pos_scores)
            for q in neg_scores
        ])
        return V10, V01

    V10_a, V01_a = placement(y_prob_a[pos_idx], y_prob_a[neg_idx])
    V10_b, V01_b = placement(y_prob_b[pos_idx], y_prob_b[neg_idx])

    auc_a = V10_a.mean()
    auc_b = V10_b.mean()

    # 2×2 covariance matrix S of (AUC_a, AUC_b)
    s10 = np.cov(np.stack([V10_a, V10_b]), ddof=1) / n1   # shape (2,2)
    s01 = np.cov(np.stack([V01_a, V01_b]), ddof=1) / n0

    S = s10 + s01   # full covariance of the AUC vector

    var_diff = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var_diff <= 0:
        return 0.0, 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return float(z), float(p)


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """
    Expected Calibration Error (ECE).
    ECE = Σ_b (|b| / N) * |acc(b) - conf(b)|
    where acc = fraction positive in bin, conf = mean predicted probability.
    """
    bins   = np.linspace(0, 1, n_bins + 1)
    ece    = 0.0
    n      = len(y_true)
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)


def hosmer_lemeshow(y_true, y_prob, n_bins=10):
    """
    Hosmer-Lemeshow goodness-of-fit test.
    Splits predictions into n_bins deciles, computes χ² statistic.
    H0: model is well-calibrated (p > 0.05 = no evidence of miscalibration).
    Returns (chi2_stat, p_value, df).
    """
    df_hl = pd.DataFrame({'y': y_true, 'p': y_prob})
    df_hl['decile'] = pd.qcut(df_hl['p'], q=n_bins, duplicates='drop')
    grouped = df_hl.groupby('decile', observed=True)

    chi2 = 0.0
    valid_bins = 0
    for _, g in grouped:
        n_g     = len(g)
        obs_pos = g['y'].sum()
        exp_pos = g['p'].sum()
        obs_neg = n_g - obs_pos
        exp_neg = n_g - exp_pos
        if exp_pos > 0 and exp_neg > 0:
            chi2 += (obs_pos - exp_pos) ** 2 / exp_pos
            chi2 += (obs_neg - exp_neg) ** 2 / exp_neg
            valid_bins += 1

    df_stat = max(valid_bins - 2, 1)
    p_val   = 1 - stats.chi2.cdf(chi2, df=df_stat)
    return float(chi2), float(p_val), df_stat


def platt_calibrate(y_true_cal, y_prob_cal, y_prob_target):
    """
    Platt scaling: fit sigmoid on cal set, apply to target.
    BUG NOTE: in the previous codebase this was fitted on the same val set
    used for model selection — here we keep them separate (val = early stop,
    test = evaluation) so calibration is fit on val, applied to test.
    """
    lr = LogisticRegression(max_iter=1000)
    lr.fit(y_prob_cal.reshape(-1, 1), y_true_cal)
    return lr.predict_proba(y_prob_target.reshape(-1, 1))[:, 1]


# =============================================================================
# STORE RAW PREDICTIONS (val for calibration fitting, test for final eval)
# =============================================================================
raw_preds_val  = {}   # val-set probabilities — used to fit Platt calibration
raw_preds_test = {}   # test-set probabilities — final evaluation


# =============================================================================
# 1. LOGISTIC REGRESSION
# =============================================================================
print("\n" + "="*60)
print("Training Logistic Regression...")
scaler_lr    = SklearnScaler()
X_train_lr   = scaler_lr.fit_transform(X_train)
X_val_lr     = scaler_lr.transform(X_val)
X_test_lr    = scaler_lr.transform(X_test)

lr_model = LogisticRegression(
    class_weight='balanced', max_iter=1000,
    solver='saga', random_state=42, n_jobs=-1
)
lr_model.fit(X_train_lr, y_train)

raw_preds_val['Logistic Regression']  = lr_model.predict_proba(X_val_lr)[:, 1]
raw_preds_test['Logistic Regression'] = lr_model.predict_proba(X_test_lr)[:, 1]

results['Logistic Regression'] = compute_metrics(y_test, raw_preds_test['Logistic Regression'])
print(f"  AUC={results['Logistic Regression']['auc']:.4f} | "
      f"AUPRC={results['Logistic Regression']['auprc']:.4f} | "
      f"Brier={results['Logistic Regression']['brier']:.4f} | "
      f"F1={results['Logistic Regression']['f1']:.4f}")


# =============================================================================
# 2. RANDOM FOREST
# =============================================================================
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_split=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)

raw_preds_val['Random Forest']  = rf_model.predict_proba(X_val)[:, 1]
raw_preds_test['Random Forest'] = rf_model.predict_proba(X_test)[:, 1]

results['Random Forest'] = compute_metrics(y_test, raw_preds_test['Random Forest'])
print(f"  AUC={results['Random Forest']['auc']:.4f} | "
      f"AUPRC={results['Random Forest']['auprc']:.4f} | "
      f"Brier={results['Random Forest']['brier']:.4f} | "
      f"F1={results['Random Forest']['f1']:.4f}")


# =============================================================================
# 3. XGBOOST  (GPU)
# =============================================================================
print("\nTraining XGBoost...")
xgb_params = {
    'objective'        : 'binary:logistic',
    'eval_metric'      : ['auc', 'aucpr'],
    'max_depth'        : 7,
    'learning_rate'    : 0.03,
    'subsample'        : 0.8,
    'colsample_bytree' : 0.8,
    'min_child_weight' : 5,
    'scale_pos_weight' : pos_weight,
    'tree_method'      : 'hist',
    'device'           : 'cuda',      # GPU
    'random_state'     : 42,
    'n_jobs'           : -1,
}

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES)
dval   = xgb.DMatrix(X_val,   label=y_val,   feature_names=FEATURES)
dtest  = xgb.DMatrix(X_test,  label=y_test,  feature_names=FEATURES)

model_xgb = xgb.train(
    xgb_params, dtrain,
    num_boost_round=3000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=100,
    verbose_eval=200
)

raw_preds_val['XGBoost']  = model_xgb.predict(dval)
raw_preds_test['XGBoost'] = model_xgb.predict(dtest)

results['XGBoost'] = compute_metrics(y_test, raw_preds_test['XGBoost'])
print(f"  Best round={model_xgb.best_iteration} | "
      f"AUC={results['XGBoost']['auc']:.4f} | "
      f"AUPRC={results['XGBoost']['auprc']:.4f} | "
      f"Brier={results['XGBoost']['brier']:.4f} | "
      f"F1={results['XGBoost']['f1']:.4f}")


# =============================================================================
# 4. LIGHTGBM  (GPU)
# =============================================================================
print("\nTraining LightGBM...")
lgb_params = {
    'objective': 'binary',
    'metric': ['auc', 'aucpr'],
    'learning_rate': 0.025,          # slightly lower for stability
    'max_depth': 6,                  # reduced from 7
    'num_leaves': 48,                # controlled
    'subsample': 0.75,
    'colsample_bytree': 0.75,
    
    # Important stability parameters
    'min_child_samples': 80,         # increased
    'min_data_in_leaf': 60,          # critical
    'min_split_gain': 0.02,          # prevents useless splits
    'lambda_l1': 1.0,                # L1 regularization
    'lambda_l2': 3.0,                # L2 regularization
    
    # Class weighting
    'scale_pos_weight': pos_weight,  # ← your desired 4.54
    'is_unbalance': False,           # use scale_pos_weight instead
    
    'device': 'gpu',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'force_row_wise': True,          # helps with GPU stability
}

train_lgb = lgb.Dataset(X_train, label=y_train)
val_lgb   = lgb.Dataset(X_val, label=y_val, reference=train_lgb)

model_lgb = lgb.train(
    lgb_params, 
    train_lgb,
    num_boost_round=5000,
    valid_sets=[train_lgb, val_lgb],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(150),     # increased patience
        lgb.log_evaluation(200)
    ]
)

raw_preds_val['LightGBM']  = model_lgb.predict(X_val)
raw_preds_test['LightGBM'] = model_lgb.predict(X_test)

results['LightGBM'] = compute_metrics(y_test, raw_preds_test['LightGBM'])
print(f"  Best round={model_lgb.best_iteration} | "
      f"AUC={results['LightGBM']['auc']:.4f} | "
      f"AUPRC={results['LightGBM']['auprc']:.4f} | "
      f"Brier={results['LightGBM']['brier']:.4f} | "
      f"F1={results['LightGBM']['f1']:.4f}")

# =============================================================================
# 5. CATBOOST  (GPU)
# =============================================================================
print("\nTraining CatBoost...")
X_train_cb = train_df[FEATURES].copy()
X_val_cb   = val_df[FEATURES].copy()
X_test_cb  = test_df[FEATURES].copy()
for col in CAT_FEATURES:
    X_train_cb[col] = X_train_cb[col].fillna('unknown').astype(str)
    X_val_cb[col]   = X_val_cb[col].fillna('unknown').astype(str)
    X_test_cb[col]  = X_test_cb[col].fillna('unknown').astype(str)

cb_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.03,
    depth=7,
    l2_leaf_reg=3,
    scale_pos_weight=pos_weight,
    cat_features=CAT_FEATURES,
    eval_metric='AUC',
    early_stopping_rounds=100,
    random_seed=42,
    verbose=200,
    task_type='GPU',   # GPU
)
cb_model.fit(
    X_train_cb, y_train,
    eval_set=(X_val_cb, y_val),
    use_best_model=True
)

raw_preds_val['CatBoost']  = cb_model.predict_proba(X_val_cb)[:, 1]
raw_preds_test['CatBoost'] = cb_model.predict_proba(X_test_cb)[:, 1]

results['CatBoost'] = compute_metrics(y_test, raw_preds_test['CatBoost'])
print(f"  AUC={results['CatBoost']['auc']:.4f} | "
      f"AUPRC={results['CatBoost']['auprc']:.4f} | "
      f"Brier={results['CatBoost']['brier']:.4f} | "
      f"F1={results['CatBoost']['f1']:.4f}")


# =============================================================================
# 6. MLP
# =============================================================================
print("\nTraining MLP...")
scaler_mlp   = SklearnScaler()
X_train_mlp  = scaler_mlp.fit_transform(X_train.values.astype(np.float32))
X_val_mlp    = scaler_mlp.transform(X_val.values.astype(np.float32))
X_test_mlp   = scaler_mlp.transform(X_test.values.astype(np.float32))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"  MLP device → {device}")

train_tensor = TensorDataset(
    torch.tensor(X_train_mlp),
    torch.tensor(y_train, dtype=torch.float32)
)
val_tensor = TensorDataset(
    torch.tensor(X_val_mlp),
    torch.tensor(y_val, dtype=torch.float32)
)
test_tensor = TensorDataset(
    torch.tensor(X_test_mlp),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_tensor, batch_size=4096, shuffle=True)
val_loader   = DataLoader(val_tensor,   batch_size=4096, shuffle=False)
test_loader  = DataLoader(test_tensor,  batch_size=4096, shuffle=False)


class AMR_MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),        nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),          nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)


mlp             = AMR_MLP(X_train_mlp.shape[1]).to(device)
criterion       = nn.BCELoss()
optimizer       = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
best_val_auc    = 0.0
best_weights    = None
patience_counter = 0
PATIENCE        = 10

for epoch in range(100):
    mlp.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(mlp(xb), yb).backward()
        optimizer.step()

    mlp.eval()
    val_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            val_preds.append(mlp(xb.to(device)).cpu().numpy())
    val_preds = np.concatenate(val_preds)
    val_auc   = roc_auc_score(y_val, val_preds)

    if val_auc > best_val_auc:
        best_val_auc     = val_auc
        best_weights     = {k: v.cpu().clone() for k, v in mlp.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d} — Val AUC: {val_auc:.4f}")

mlp.load_state_dict(best_weights)
mlp.eval()

# val preds (for Platt calibration fitting)
raw_preds_val['MLP'] = np.concatenate([
    mlp(torch.tensor(xb, dtype=torch.float32).to(device)).cpu().detach().numpy()
    for xb, _ in val_loader
])
# test preds (final evaluation)
raw_preds_test['MLP'] = np.concatenate([
    mlp(torch.tensor(xb, dtype=torch.float32).to(device)).cpu().detach().numpy()
    for xb, _ in test_loader
])

results['MLP'] = compute_metrics(y_test, raw_preds_test['MLP'])
print(f"  AUC={results['MLP']['auc']:.4f} | "
      f"AUPRC={results['MLP']['auprc']:.4f} | "
      f"Brier={results['MLP']['brier']:.4f} | "
      f"F1={results['MLP']['f1']:.4f}")


# =============================================================================
# SUMMARY TABLE (test set)
# =============================================================================
print("\n" + "="*95)
print(f"{'Model':<22} {'AUC':>7} {'AUPRC':>7} {'Brier':>7} {'BSS':>7} "
      f"{'Precision':>10} {'Recall':>8} {'F1':>7}")
print("="*95)
for name, m in results.items():
    print(f"{name:<22} {m['auc']:>7.4f} {m['auprc']:>7.4f} {m['brier']:>7.4f} "
          f"{m['bss']:>7.4f} {m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>7.4f}")
print("="*95)
print(f"  Baseline Brier (prevalence²) : {y_test.mean()*(1-y_test.mean()):.4f}")
print(f"  Test prevalence              : {y_test.mean():.4f}")


# =============================================================================
# BOOTSTRAP CONFIDENCE INTERVALS  (1 000 resamples, test set)
# =============================================================================
print("\n--- Bootstrap 95% CIs on test set (n=1000) ---")
ci_rows = []
for name, y_prob in raw_preds_test.items():
    auc_m,  auc_lo,  auc_hi  = bootstrap_ci(y_test, y_prob, roc_auc_score)
    brier_m, brier_lo, brier_hi = bootstrap_ci(
        y_test, y_prob, brier_score_loss)
    ci_rows.append({
        'Model'        : name,
        'AUC'          : f"{auc_m:.4f} [{auc_lo:.4f}–{auc_hi:.4f}]",
        'Brier'        : f"{brier_m:.4f} [{brier_lo:.4f}–{brier_hi:.4f}]",
    })
    print(f"  {name:<22}  AUC {auc_m:.4f} [{auc_lo:.4f}–{auc_hi:.4f}]  "
          f"Brier {brier_m:.4f} [{brier_lo:.4f}–{brier_hi:.4f}]")

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv('/kaggle/working/model_ci_results.csv', index=False)



# =============================================================================
# DELONG PAIRWISE TESTS  (test set — all pairs)
# =============================================================================
from itertools import combinations

print("\n--- DeLong pairwise tests (test set) ---")
print(f"{'Model A':<22} vs {'Model B':<22}  ΔAUC     Z       p-value  Sig   Practical")
print("-"*95)

MEANINGFUL_DELTA = 0.02  # threshold for practical significance

delong_rows = []
model_names  = list(raw_preds_test.keys())

for m_a, m_b in combinations(model_names, 2):
    z, p  = delong_test(y_test, raw_preds_test[m_a], raw_preds_test[m_b])
    delta = results[m_a]['auc'] - results[m_b]['auc']

    # Statistical significance
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    # Practical significance — is the difference large enough to matter clinically?
    if abs(delta) >= MEANINGFUL_DELTA and p < 0.05:
        practical = 'meaningful'
    elif abs(delta) < MEANINGFUL_DELTA and p < 0.05:
        practical = 'trivial'     # statistically sig but clinically negligible
    else:
        practical = 'n/a'

    print(f"{m_a:<22} vs {m_b:<22}  {delta:+.4f}  {z:+7.3f}  {p:.4f}   {sig:<5} {practical}")
    delong_rows.append({
        'Model_A'      : m_a,
        'Model_B'      : m_b,
        'delta_AUC'    : round(delta, 5),
        'Z'            : round(z, 4),
        'p_value'      : round(p, 6),
        'Sig'          : sig,
        'Practical'    : practical,
    })

print("Significance : *** p<0.001  ** p<0.01  * p<0.05  ns p≥0.05")
print(f"Practical    : meaningful = |ΔAUC| ≥ {MEANINGFUL_DELTA} AND p<0.05 | "
      f"trivial = statistically sig but |ΔAUC| < {MEANINGFUL_DELTA}")

delong_df = pd.DataFrame(delong_rows)
delong_df.to_csv('/kaggle/working/delong_test_results.csv', index=False)
print("\n✅ Saved: delong_test_results.csv")

# =============================================================================
# CALIBRATION METRICS  —  ECE + Hosmer-Lemeshow  (test set, raw probabilities)
# =============================================================================
print("\n--- Calibration metrics (test set, raw probabilities) ---")
print(f"{'Model':<22}  {'ECE':>7}  {'HL χ²':>9}  {'HL p':>8}  {'Interpretation'}")
print("-"*75)

cal_rows = []
for name, y_prob in raw_preds_test.items():
    ece          = expected_calibration_error(y_test, y_prob, n_bins=15)
    hl_chi2, hl_p, hl_df = hosmer_lemeshow(y_test, y_prob, n_bins=10)
    interp = "well-calibrated" if hl_p > 0.05 else "MISCALIBRATED"
    print(f"  {name:<22}  {ece:.4f}   {hl_chi2:>9.2f}   {hl_p:.4f}  {interp}")
    cal_rows.append({
        'Model': name, 'ECE': round(ece, 5),
        'HL_chi2': round(hl_chi2, 3), 'HL_p': round(hl_p, 4),
        'Calibrated': hl_p > 0.05
    })

cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv('/kaggle/working/calibration_metrics.csv', index=False)
print("\nNote: HL H0 = well-calibrated. p > 0.05 → no evidence of miscalibration.")
print("ECE close to 0 → predicted probabilities match observed resistance rates.")


# =============================================================================
# POST-HOC CALIBRATION  —  Platt scaling fitted on val, evaluated on test
# Fitted on val set (which was NOT used for final metric reporting) so that
# test set remains fully held-out.
# =============================================================================
print("\n--- Post-hoc Platt calibration (fit on val, eval on test) ---")
cal_test_preds = {}

for name in raw_preds_test:
    cal_prob = platt_calibrate(
        y_val, raw_preds_val[name], raw_preds_test[name]
    )
    cal_test_preds[name] = cal_prob
    ece_before = expected_calibration_error(y_test, raw_preds_test[name])
    ece_after  = expected_calibration_error(y_test, cal_prob)
    bs_before  = brier_score_loss(y_test, raw_preds_test[name])
    bs_after   = brier_score_loss(y_test, cal_prob)
    print(f"  {name:<22}  ECE {ece_before:.4f}→{ece_after:.4f}  "
          f"Brier {bs_before:.4f}→{bs_after:.4f}")


# =============================================================================
# RELIABILITY DIAGRAMS  (6 models × 2 columns = 3-row figure)
# =============================================================================
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
fig.patch.set_facecolor('white')
axes = axes.flatten()

BLUE = '#185FA5'; GRAY = '#888780'; RED = '#A32D2D'
prevalence_test = y_test.mean()
bs_ref = prevalence_test * (1 - prevalence_test)

for ax, (name, y_prob) in zip(axes, raw_preds_test.items()):
    # calibration curve
    frac_pos, mean_pred = calibration_curve(y_test, y_prob, n_bins=15, strategy='uniform')
    cal_prob = cal_test_preds[name]
    frac_pos_cal, mean_pred_cal = calibration_curve(
        y_test, cal_prob, n_bins=15, strategy='uniform')

    ece  = expected_calibration_error(y_test, y_prob)
    ece_cal = expected_calibration_error(y_test, cal_prob)
    bs   = brier_score_loss(y_test, y_prob)

    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Perfect')
    ax.plot(mean_pred,     frac_pos,     'o-', color=RED,  lw=2, ms=5,
            label=f'Raw  ECE={ece:.3f}')
    ax.plot(mean_pred_cal, frac_pos_cal, 's--', color=BLUE, lw=1.5, ms=4,
            label=f'Platt ECE={ece_cal:.3f}')

    # histogram of predicted probabilities (density, secondary axis)
    ax2 = ax.twinx()
    ax2.hist(y_prob, bins=30, alpha=0.12, color=GRAY, density=True)
    ax2.set_ylabel('Density', fontsize=8, color=GRAY)
    ax2.tick_params(axis='y', labelsize=7, colors=GRAY)
    ax2.set_ylim(0, ax2.get_ylim()[1] * 5)

    ax.set_xlabel('Mean predicted probability', fontsize=9)
    ax.set_ylabel('Fraction of positives',      fontsize=9)
    ax.set_title(f'{name}  (Brier={bs:.4f}  BSS={results[name]["bss"]:.3f})',
                 fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, frameon=False, loc='upper left')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.spines['top'].set_visible(False)

plt.suptitle('Reliability Diagrams — Test Set (>2020)\n'
             'Red = raw probabilities | Blue = Platt-calibrated | '
             'Diagonal = perfect calibration',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_reliability_diagrams.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_reliability_diagrams.png")


# =============================================================================
# SUMMARY COMPARISON FIGURE  (AUC + Brier + ECE, test set, with CI error bars)
# =============================================================================
model_list = list(raw_preds_test.keys())
auc_vals   = [results[m]['auc']   for m in model_list]
brier_vals = [results[m]['brier'] for m in model_list]
ece_vals   = [expected_calibration_error(y_test, raw_preds_test[m]) for m in model_list]

# collect CI bounds for AUC error bars
auc_lo, auc_hi = [], []
for m in model_list:
    _, lo, hi = bootstrap_ci(y_test, raw_preds_test[m], roc_auc_score, n_boot=500)
    auc_lo.append(lo); auc_hi.append(hi)

auc_err = [
    [a - lo for a, lo in zip(auc_vals, auc_lo)],
    [hi - a for a, hi in zip(auc_vals, auc_hi)]
]

fig, axes = plt.subplots(1, 3, figsize=(17, 6))
fig.patch.set_facecolor('white')

# AUC with CIs
ax = axes[0]
colors = [BLUE if m == 'LightGBM' else '#B5D4F4' for m in model_list]
bars = ax.bar(model_list, auc_vals, color=colors, width=0.6,
              edgecolor='white', yerr=auc_err, capsize=4,
              error_kw=dict(ecolor=GRAY, lw=1.5))
ax.axhline(0.5, color=GRAY, ls='--', lw=1, alpha=0.6, label='Random (0.5)')
ax.set_ylim(min(auc_vals) - 0.06, 1.0)
ax.set_ylabel('ROC-AUC', fontsize=11)
ax.set_title('ROC-AUC with 95% CI\n(bootstrap, test set)', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=9, frameon=False)
for bar, val in zip(bars, auc_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.004,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# Brier score
ax = axes[1]
colors_b = [RED if m == 'LightGBM' else '#F5B8B8' for m in model_list]
bars = ax.bar(model_list, brier_vals, color=colors_b, width=0.6, edgecolor='white')
ax.axhline(bs_ref, color=GRAY, ls='--', lw=1, alpha=0.6,
           label=f'Prevalence baseline ({bs_ref:.4f})')
ax.set_ylabel('Brier Score  (↓ better)', fontsize=11)
ax.set_title('Brier Score\n(test set)', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.legend(fontsize=9, frameon=False)
for bar, val in zip(bars, brier_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

# ECE
ax = axes[2]
colors_e = ['#3B6D11' if m == 'LightGBM' else '#B8D9A0' for m in model_list]
bars = ax.bar(model_list, ece_vals, color=colors_e, width=0.6, edgecolor='white')
ax.set_ylabel('ECE  (↓ better)', fontsize=11)
ax.set_title('Expected Calibration Error\n(test set, 15 bins)', fontsize=11, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, ece_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Model Comparison — Test Set (>2020)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/fig_model_comparison_full.png', dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_model_comparison_full.png")


# =============================================================================
# SAVE MODELS + ARTEFACTS
# =============================================================================
model_xgb.save_model('/kaggle/working/xgboost_model.json')
model_lgb.save_model('/kaggle/working/lightgbm_model.txt')
cb_model.save_model('/kaggle/working/catboost_model.cbm')
torch.save(best_weights, '/kaggle/working/mlp_weights.pt')

with open('/kaggle/working/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
with open('/kaggle/working/scaler_mlp.pkl', 'wb') as f:
    pickle.dump(scaler_mlp, f)
with open('/kaggle/working/scaler_lr.pkl', 'wb') as f:
    pickle.dump(scaler_lr, f)

# save raw test predictions for downstream LOCO / KZ inference
preds_out = pd.DataFrame(raw_preds_test)
preds_out['y_true'] = y_test
preds_out.to_csv('/kaggle/working/test_predictions.csv', index=False)

print("\n✅ All models and artefacts saved.")
print(f"\nFinal test-set metrics summary:")
print(ci_df.to_string(index=False))

Train : 11,207,215 rows (≤ 2018)
Val   : 3,607,997 rows (2019–2020)  [early stopping only]
Test  : 5,631,656 rows (> 2020)  [final reported metrics]

scale_pos_weight : 4.72
Test class dist  : 1,081,295 resistant / 4,550,361 susceptible

Training Logistic Regression...
  AUC=0.6494 | AUPRC=0.2957 | Brier=0.2421 | F1=0.3755

Training Random Forest...
  AUC=0.8285 | AUPRC=0.5647 | Brier=0.1735 | F1=0.5197

Training XGBoost...
[0]	train-auc:0.79943	train-aucpr:0.54009	val-auc:0.77029	val-aucpr:0.49703
[200]	train-auc:0.89137	train-aucpr:0.67481	val-auc:0.86458	val-aucpr:0.62755
[400]	train-auc:0.90478	train-aucpr:0.70622	val-auc:0.87758	val-aucpr:0.65721
[600]	train-auc:0.90961	train-aucpr:0.71800	val-auc:0.88161	val-aucpr:0.66767
[800]	train-auc:0.91229	train-aucpr:0.72463	val-auc:0.88387	val-aucpr:0.67416
[1000]	train-auc:0.91408	train-aucpr:0.72919	val-auc:0.88546	val-aucpr:0.67849
[1200]	train-auc:0.91527	train-aucpr:0.73224	val-auc:0.88634	val-aucpr:0.68088
[1400]	train-auc:0.91629	t

1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 150 rounds
[200]	train's auc: 0.878993	val's auc: 0.853547
[400]	train's auc: 0.893873	val's auc: 0.868852
[600]	train's auc: 0.900163	val's auc: 0.873963
[800]	train's auc: 0.904056	val's auc: 0.877076
[1000]	train's auc: 0.906784	val's auc: 0.879251
[1200]	train's auc: 0.908741	val's auc: 0.880863
[1400]	train's auc: 0.910145	val's auc: 0.882
[1600]	train's auc: 0.91122	val's auc: 0.882891
[1800]	train's auc: 0.912024	val's auc: 0.88333
[2000]	train's auc: 0.91291	val's auc: 0.884015
[2200]	train's auc: 0.913514	val's auc: 0.884483
[2400]	train's auc: 0.914037	val's auc: 0.884725
[2600]	train's auc: 0.914495	val's auc: 0.884865
[2800]	train's auc: 0.91495	val's auc: 0.885055
[3000]	train's auc: 0.915308	val's auc: 0.885023
Early stopping, best iteration is:
[2884]	train's auc: 0.915113	val's auc: 0.885121
  Best round=2884 | AUC=0.8483 | AUPRC=0.6146 | Brier=0.1613 | F1=0.5436

Training CatBoost...


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8450013	best: 0.8450013 (0)	total: 1.02s	remaining: 1h 25m 7s
200:	test: 0.8796773	best: 0.8796773 (200)	total: 3m 18s	remaining: 1h 18m 55s
400:	test: 0.8829795	best: 0.8829891 (398)	total: 6m 21s	remaining: 1h 12m 51s
600:	test: 0.8844439	best: 0.8844439 (600)	total: 9m 20s	remaining: 1h 8m 24s
800:	test: 0.8849376	best: 0.8849376 (800)	total: 12m 20s	remaining: 1h 4m 44s
1000:	test: 0.8857405	best: 0.8857405 (1000)	total: 15m 28s	remaining: 1h 1m 50s
1200:	test: 0.8862187	best: 0.8862187 (1200)	total: 18m 36s	remaining: 58m 50s
1400:	test: 0.8863798	best: 0.8863798 (1400)	total: 21m 47s	remaining: 55m 57s
bestTest = 0.8866202831
bestIteration = 1488
Shrink model to first 1489 iterations.
  AUC=0.8573 | AUPRC=0.6497 | Brier=0.1536 | F1=0.5490

Training MLP...


In [ ]:
# =============================================================================
# BOOTSTRAP 95% CI  —  4 metrics × 6 models  (2000 resamples)
# =============================================================================
N_BOOT = 2000

METRIC_FNS = {
    'AUC'  : roc_auc_score,
    'AUPRC': average_precision_score,
    'F1'   : lambda yt, yp: f1_score(yt, (yp >= 0.5).astype(int), zero_division=0),
    'Brier': brier_score_loss,
}

# --- collect bootstrap distributions (store full arrays for violin plots) ---
boot_distributions = {metric: {} for metric in METRIC_FNS}
boot_summary       = []   # for the table / error-bar plot

rng = np.random.default_rng(42)
n   = len(y_test)

print("\n--- Bootstrap 95% CI (n=2000 resamples) ---")
for name, y_prob in raw_preds_test.items():
    row = {'Model': name}
    for metric, fn in METRIC_FNS.items():
        vals = []
        for _ in range(N_BOOT):
            idx = rng.integers(0, n, n)
            if len(np.unique(y_test[idx])) < 2:
                continue
            try:
                vals.append(fn(y_test[idx], y_prob[idx]))
            except Exception:
                continue
        vals = np.array(vals)
        lo, hi = np.percentile(vals, [2.5, 97.5])
        mid    = vals.mean()
        boot_distributions[metric][name] = vals
        row[f'{metric}_mean'] = mid
        row[f'{metric}_lo']   = lo
        row[f'{metric}_hi']   = hi
        print(f"  {name:<22} {metric:<6}  {mid:.4f}  [{lo:.4f} – {hi:.4f}]")
    boot_summary.append(row)

boot_df = pd.DataFrame(boot_summary)
boot_df.to_csv('/kaggle/working/bootstrap_ci_results.csv', index=False)
print("✅ Saved: bootstrap_ci_results.csv")


# =============================================================================
# FIGURE — 2 × 2 grid, one panel per metric
# forest-plot style (horizontal, models on y-axis, CI as whiskers)
# =============================================================================
MODEL_ORDER = list(raw_preds_test.keys())   # keep insertion order
COLORS = {
    'Logistic Regression': '#B5D4F4',
    'Random Forest'      : '#9FE1CB',
    'XGBoost'            : '#FAC775',
    'LightGBM'           : '#5DCAA5',
    'CatBoost'           : '#185FA5',
    'MLP'                : '#F0997B',
}
LOWER_BETTER = {'Brier', 'ECE'}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.patch.set_facecolor('white')
axes = axes.flatten()

for ax, (metric, fn) in zip(axes, METRIC_FNS.items()):
    y_pos   = np.arange(len(MODEL_ORDER))
    means   = [boot_df.loc[boot_df['Model'] == m, f'{metric}_mean'].values[0] for m in MODEL_ORDER]
    los     = [boot_df.loc[boot_df['Model'] == m, f'{metric}_lo'].values[0]   for m in MODEL_ORDER]
    his     = [boot_df.loc[boot_df['Model'] == m, f'{metric}_hi'].values[0]   for m in MODEL_ORDER]
    xerr    = [[m - lo for m, lo in zip(means, los)],
               [hi - m for m, hi in zip(means, his)]]
    colors  = [COLORS[m] for m in MODEL_ORDER]

    # find best model (min if lower-better, else max)
    best_idx = int(np.argmin(means)) if metric in LOWER_BETTER else int(np.argmax(means))

    bars = ax.barh(y_pos, means, xerr=xerr, color=colors,
                   height=0.55, capsize=5,
                   error_kw=dict(ecolor='#444441', lw=1.5, capthick=1.5),
                   edgecolor='white')

    # bold outline on best model
    bars[best_idx].set_edgecolor('#2C2C2A')
    bars[best_idx].set_linewidth(1.8)

    # value labels
    for i, (m, lo, hi) in enumerate(zip(means, los, his)):
        ax.text(hi + (max(his) - min(los)) * 0.01,
                y_pos[i], f'{m:.4f}', va='center', fontsize=8.5,
                color='#2C2C2A')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(MODEL_ORDER, fontsize=10)
    arrow = '↓ better' if metric in LOWER_BETTER else '↑ better'
    ax.set_xlabel(f'{metric}  ({arrow})', fontsize=10)
    ax.set_title(f'{metric} — 95% CI (bootstrap, n={N_BOOT})',
                 fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # subtle grid
    ax.xaxis.grid(True, linestyle='--', alpha=0.4, lw=0.8)
    ax.set_axisbelow(True)

plt.suptitle('Bootstrap 95% Confidence Intervals — Test Set (>2020)\n'
             'Bold outline = best performing model per metric',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/fig_bootstrap_ci_4metrics.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_bootstrap_ci_4metrics.png")


# =============================================================================
# FIGURE 2 — violin/density of bootstrap distributions (AUC only, all 6 models)
# shows the full shape of uncertainty, not just the interval endpoints
# =============================================================================
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('white')

positions = np.arange(len(MODEL_ORDER))
parts = ax.violinplot(
    [boot_distributions['AUC'][m] for m in MODEL_ORDER],
    positions=positions, widths=0.6, showmedians=True,
    showextrema=False
)

for i, (body, m) in enumerate(zip(parts['cmedians'].get_segments(), MODEL_ORDER)):
    pass

for i, (pc, m) in enumerate(zip(parts['bodies'], MODEL_ORDER)):
    pc.set_facecolor(COLORS[m])
    pc.set_edgecolor('#444441')
    pc.set_alpha(0.85)
    pc.set_linewidth(0.8)

parts['cmedians'].set_color('#2C2C2A')
parts['cmedians'].set_linewidth(2)

# overlay CI dots
for i, m in enumerate(MODEL_ORDER):
    mid = boot_df.loc[boot_df['Model'] == m, 'AUC_mean'].values[0]
    ax.scatter(i, mid, color='white', s=40, zorder=5, edgecolors='#2C2C2A', lw=1.2)

ax.set_xticks(positions)
ax.set_xticklabels(MODEL_ORDER, fontsize=10)
ax.set_ylabel('AUC (bootstrap distribution)', fontsize=10)
ax.set_title('AUC Bootstrap Distribution — All 6 Models\n'
             'Violin = full 2000-resample distribution | dot = mean | bar = median',
             fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, linestyle='--', alpha=0.4, lw=0.8)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('/kaggle/working/fig_bootstrap_auc_violin.png',
            dpi=300, bbox_inches='tight')
plt.show()
print("✅ Saved: fig_bootstrap_auc_violin.png")

# BOOTSTRAP DATA NEEDED (ABOVE)

In [ ]:
import zipfile
import os

exclude_files = {'combined_dataset.csv'}
working_dir = '/kaggle/working'

with zipfile.ZipFile('/kaggle/working/models_and_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(working_dir):
        if fname in exclude_files or fname == 'models_and_results.zip':
            continue
        fpath = os.path.join(working_dir, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, fname)
            print(f"  Added: {fname}")

print("\nDone. Download: /kaggle/working/models_and_results.zip")

## 10. Kazakhstan Macroeconomic Data Processing

Kazakhstan-specific antibiotic consumption (DID per 1000/day, 2017–2023) merged with Kazakhstan World Bank macro indicators.
This dataset is used **exclusively for zero-shot inference** — never seen during model training.


In [5]:
SANITATION_PATH    = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874/API_SH.STA.SMSS.ZS_DS2_en_csv_v2_9874.csv'
WATER_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137/API_SH.H2O.BASW.ZS_DS2_en_csv_v2_3137.csv'
HOSPITAL_BEDS_PATH = '/kaggle/input/datasets/uaisamangeldi/highfeature/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206/API_SH.MED.BEDS.ZS_DS2_en_csv_v2_2206.csv'
TEMPERATURE_PATH   = '/kaggle/input/datasets/uaisamangeldi/highfeature/average-monthly-surface-temperature/average-monthly-surface-temperature.csv'
AWARE_PATH         = '/kaggle/input/datasets/uaisamangeldi/highfeature/data.csv'

In [6]:
KZ_MACRO_FILES = {
    'gdp_per_capita_usd':         '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46/API_NY.GDP.PCAP.CD_DS2_en_csv_v2_46.csv',
    'health_expenditure_pct_gdp': '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_558/API_SH.XPD.CHEX.GD.ZS_DS2_en_csv_v2_558.csv',
    'population_total':           '/kaggle/input/datasets/uaisamangeldi/macro-kz/API_SP.POP.TOTL_DS2_en_csv_v2_61/API_SP.POP.TOTL_DS2_en_csv_v2_61.csv',
}
KZ_DID_PATH = '/kaggle/input/datasets/uaisamangeldi/kz-data/kazakhstan_antibiotics_did_2017_2023.csv'
KZ_AREA_KM2 = 2_724_900

# ── KZ AWaRe Access % (hardcoded from table) ─────────────────────────────────
KZ_AWARE = pd.DataFrame({
    'year':             [2017,  2018,  2019,  2020,  2021,  2022,  2023],
    'aware_access_pct': [33.00, 15.00, 52.21, 40.63, 40.27, 43.17, 44.33],
})

# ── KZ New WB Features (sanitation, water, hospital beds) ────────────────────
# Pull from the same global WB files used in Cell 4
KZ_SANITATION_PATH    = SANITATION_PATH
KZ_WATER_PATH         = WATER_PATH
KZ_HOSPITAL_BEDS_PATH = HOSPITAL_BEDS_PATH
KZ_TEMPERATURE_PATH   = TEMPERATURE_PATH

def load_kz_wb_indicator(path, col_name):
    df = pd.read_csv(path, skiprows=4)
    df.columns = df.columns.str.strip()
    df = df[df['Country Name'] == 'Kazakhstan'].copy()
    year_cols = [c for c in df.columns if c.strip().isdigit() and int(c.strip()) >= 2000]
    df = df.melt(id_vars=['Country Name'], value_vars=year_cols, var_name='year', value_name=col_name)
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    df = df.dropna(subset=['year', col_name])
    df['year'] = df['year'].astype(int)
    return df[['year', col_name]]

# ── World Bank macros ─────────────────────────────────────────────────────────
kz_macro = load_kz_wb_indicator(KZ_MACRO_FILES['gdp_per_capita_usd'], 'gdp_per_capita_usd')
for col, path in list(KZ_MACRO_FILES.items())[1:]:
    kz_macro = kz_macro.merge(load_kz_wb_indicator(path, col), on='year', how='outer')

kz_macro = kz_macro.sort_values('year').reset_index(drop=True)
kz_macro['population_density_per_sq_km'] = kz_macro['population_total'] / KZ_AREA_KM2

# ── Merge new WB features into kz_macro ──────────────────────────────────────
for path, name in [
    (KZ_SANITATION_PATH,    'sanitation_access_pct'),
    (KZ_WATER_PATH,         'water_access_pct'),
    (KZ_HOSPITAL_BEDS_PATH, 'hospital_beds_per_1000'),
]:
    try:
        df_tmp = load_kz_wb_indicator(path, name)
        kz_macro = kz_macro.merge(df_tmp, on='year', how='left')
        print(f" {name}: OK")
    except Exception as e:
        print(f" {name}: FAILED ({e})")
        kz_macro[name] = np.nan

# ── Merge KZ temperature ──────────────────────────────────────────────────────
try:
    df_temp = pd.read_csv(KZ_TEMPERATURE_PATH, usecols=['Entity', 'Day', 'Monthly average'])
    df_temp = df_temp[df_temp['Entity'] == 'Kazakhstan'].copy()
    df_temp['year'] = pd.to_datetime(df_temp['Day'], errors='coerce').dt.year
    kz_temp_yearly = (df_temp.groupby('year', as_index=False)['Monthly average']
                              .mean()
                              .rename(columns={'Monthly average': 'mean_temp_celsius'}))
    del df_temp; gc.collect()
    kz_macro = kz_macro.merge(kz_temp_yearly, on='year', how='left')
    print(" mean_temp_celsius: OK")
except Exception as e:
    print(f" mean_temp_celsius: FAILED ({e})")
    kz_macro['mean_temp_celsius'] = np.nan

# ── Merge AWaRe ───────────────────────────────────────────────────────────────
kz_macro = kz_macro.merge(KZ_AWARE, on='year', how='left')

# ── Load KZ DID ───────────────────────────────────────────────────────────────
kz_did = pd.read_csv(KZ_DID_PATH)
kz_did.columns = kz_did.columns.str.strip()
kz_did['antibiotic']     = kz_did['antibiotic'].str.strip().str.lower().str.replace('/', ' / ', regex=False)
kz_did['aware_category'] = kz_did['aware_category'].fillna('Unknown').str.strip()
kz_did['did']            = pd.to_numeric(kz_did['did'], errors='coerce').fillna(0)
kz_did['year']           = kz_did['year'].astype(int)

# ── Per-class and total consumption ──────────────────────────────────────────
kz_class_did = (
    kz_did
    .groupby(['year', 'aware_category'])['did']
    .sum()
    .reset_index()
    .rename(columns={'aware_category': 'antibiotic_class',
                     'did': 'consumption_did_per_class'})
)

kz_total_did = (
    kz_did
    .groupby('year')['did']
    .sum()
    .reset_index()
    .rename(columns={'did': 'consumption_ddd_total'})
)

# ── Filter macro to 2017–2023 and merge consumption ──────────────────────────
kz_macro_filtered = kz_macro[(kz_macro['year'] >= 2017) & (kz_macro['year'] <= 2023)].copy()
kz_macro_filtered = kz_macro_filtered.merge(kz_total_did, on='year', how='left')

# ── Impute missing new features using global means ────────────────────────────
new_feature_cols = ['sanitation_access_pct', 'water_access_pct',
                    'hospital_beds_per_1000', 'mean_temp_celsius', 'aware_access_pct']
for col in new_feature_cols:
    if col in kz_macro_filtered.columns:
        if kz_macro_filtered[col].isna().any():
            global_mean = full_df[col].mean() if col in full_df.columns else np.nan
            kz_macro_filtered[col] = kz_macro_filtered[col].fillna(global_mean)
            print(f" {col}: filled {kz_macro_filtered[col].isna().sum()} NaNs with global mean ({global_mean:.2f})")

# ── Build final KZ dataframe ──────────────────────────────────────────────────
kz_final = kz_did.merge(kz_macro_filtered, on='year', how='inner')
kz_final = kz_final.rename(columns={'aware_category': 'antibiotic_class'})
kz_final = kz_final.merge(kz_class_did, on=['year', 'antibiotic_class'], how='left')

# Fallback imputation for class-level DID
global_mean_did = np.expm1(full_df['log_consumption_did_per_class'].mean())
kz_final['consumption_did_per_class'] = kz_final['consumption_did_per_class'].fillna(global_mean_did)

# ── Log transforms ────────────────────────────────────────────────────────────
kz_final['log_gdp_per_capita']            = np.log1p(kz_final['gdp_per_capita_usd'].clip(lower=0))
kz_final['log_consumption_ddd']           = np.log1p(kz_final['consumption_ddd_total'].clip(lower=0))
kz_final['log_consumption_did_per_class'] = np.log1p(kz_final['consumption_did_per_class'].clip(lower=0))

# ── Drop raw columns ──────────────────────────────────────────────────────────
kz_final = kz_final.drop(columns=['gdp_per_capita_usd', 'consumption_ddd_total',
                                   'consumption_did_per_class', 'population_total'])
kz_final = kz_final.sort_values(['year', 'antibiotic']).reset_index(drop=True)

# ── Verify ────────────────────────────────────────────────────────────────────
log_cols = ['log_gdp_per_capita', 'log_consumption_ddd', 'log_consumption_did_per_class']
check_cols = log_cols + new_feature_cols
print(f"\nKazakhstan inference dataset: {kz_final.shape}")
print(f"\nFeature coverage (non-null %):")
print((kz_final[check_cols].notna().mean() * 100).round(1))
print(f"\nSample:")
print(kz_final[['year', 'antibiotic', 'antibiotic_class'] + log_cols].head(6).to_string(index=False))

kz_final.to_csv('/kaggle/working/kazakhstan_final_prediction_input_2017_2023.csv', index=False)
print("\nSaved.")

 sanitation_access_pct: OK
 water_access_pct: OK
 hospital_beds_per_1000: OK
 mean_temp_celsius: FAILED (name 'gc' is not defined)
 sanitation_access_pct: filled 0 NaNs with global mean (81.91)
 hospital_beds_per_1000: filled 0 NaNs with global mean (4.44)
 mean_temp_celsius: filled 0 NaNs with global mean (13.63)

Kazakhstan inference dataset: (183, 15)

Feature coverage (non-null %):
log_gdp_per_capita               100.0
log_consumption_ddd              100.0
log_consumption_did_per_class    100.0
sanitation_access_pct            100.0
water_access_pct                 100.0
hospital_beds_per_1000           100.0
mean_temp_celsius                100.0
aware_access_pct                 100.0
dtype: float64

Sample:
 year                    antibiotic antibiotic_class  log_gdp_per_capita  log_consumption_ddd  log_consumption_did_per_class
 2017                      amikacin           Access            9.098748             1.391033                       0.776109
 2017                   a

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import os
import numpy as np
country_profile = full_df.groupby('country').agg(
    log_gdp            = ('log_gdp_per_capita',           'mean'),
    health_exp         = ('health_expenditure_pct_gdp',   'mean'),
    pop_density        = ('population_density_per_sq_km', 'mean'),
    log_cons_total     = ('log_consumption_ddd',          'mean'),
    log_cons_per_class = ('log_consumption_did_per_class','mean'),
    # ── NEW FEATURES ─────────────────────────────────────
    sanitation         = ('sanitation_access_pct',        'mean'),
    water              = ('water_access_pct',             'mean'),
    hospital_beds      = ('hospital_beds_per_1000',       'mean'),
    mean_temp          = ('mean_temp_celsius',            'mean'),
    aware_access       = ('aware_access_pct',             'mean'),
).reset_index().set_index('country')

# ── Kazakhstan Profile ──────────────────────────────────────────────────────
kz_profile = pd.DataFrame([{
    'log_gdp':            kz_final['log_gdp_per_capita'].mean(),
    'health_exp':         kz_final['health_expenditure_pct_gdp'].mean(),
    'pop_density':        kz_final['population_density_per_sq_km'].mean(),
    'log_cons_total':     kz_final['log_consumption_ddd'].mean(),
    'log_cons_per_class': kz_final['log_consumption_did_per_class'].mean(),
    'sanitation':         kz_final['sanitation_access_pct'].mean(),
    'water':              kz_final['water_access_pct'].mean(),
    'hospital_beds':      kz_final['hospital_beds_per_1000'].mean(),
    'mean_temp':          kz_final['mean_temp_celsius'].mean(),
    'aware_access':       kz_final['aware_access_pct'].mean(),
}], index=['Kazakhstan'])

# Combine
country_profile = pd.concat([country_profile, kz_profile])

print(f"Total countries for clustering: {len(country_profile)}")
print(f"Any NaNs in profile:\n{country_profile.isna().sum()[country_profile.isna().sum() > 0]}")

country_profile = country_profile.fillna(country_profile.mean())

feature_cols = [
    'log_gdp', 'health_exp', 'pop_density',
    'log_cons_total', 'log_cons_per_class',
    'sanitation', 'water', 'hospital_beds',
    'mean_temp', 'aware_access'
]

# ── Scale and Cluste
scaler = StandardScaler()
X_cluster = scaler.fit_transform(country_profile[feature_cols])

kmeans = KMeans(n_clusters=6, random_state=42, n_init=20)
country_profile['cluster'] = kmeans.fit_predict(X_cluster)

# ── Kazakhstan Results
kz_cluster = country_profile.loc['Kazakhstan', 'cluster']
kz_row = country_profile.loc['Kazakhstan', feature_cols]

print(f"\nKazakhstan is in cluster {kz_cluster}")
print("\nKazakhstan feature profile:")
print(kz_row.to_string())

print("\nCountries in Kazakhstan's cluster:")
cluster_members = country_profile[country_profile['cluster'] == kz_cluster].drop(columns='cluster')
print(cluster_members.index.tolist())

kz_scaled = scaler.transform(kz_row.values.reshape(1, -1))
centroid = kmeans.cluster_centers_[kz_cluster]

distances = {}
for country in cluster_members.index:
    row = country_profile.loc[country, feature_cols].values.reshape(1, -1)
    scaled = scaler.transform(row)
    distances[country] = np.linalg.norm(scaled - centroid)

dist_df = pd.Series(distances).sort_values().rename('distance_to_centroid')
print("\nDistance to cluster centroid (lower = more similar):")
print(dist_df.to_string())

check = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
ps_df = country_profile[country_profile.index.isin(check)][['cluster']]
print("\nPost-Soviet cluster assignments:")
print(ps_df.to_string())

os.makedirs('/kaggle/working', exist_ok=True)
country_profile.to_csv('/kaggle/working/country_clusters.csv')
print("\n✅ Saved country_clusters.csv")

Total countries for clustering: 86
Any NaNs in profile:
Series([], dtype: int64)

Kazakhstan is in cluster 5

Kazakhstan feature profile:
log_gdp                9.192614
health_exp             3.309739
pop_density            7.097607
log_cons_total         2.067830
log_cons_per_class     1.418312
sanitation            81.909807
water                 96.759702
hospital_beds          5.085287
mean_temp              7.533962
aware_access          37.528907

Countries in Kazakhstan's cluster:
['Austria', 'Canada', 'Denmark', 'Finland', 'Sweden', 'Kazakhstan']

Distance to cluster centroid (lower = more similar):
Sweden        1.574596
Finland       1.784489
Denmark       1.874973
Austria       2.001884
Canada        2.626046
Kazakhstan    4.459348

Post-Soviet cluster assignments:
           cluster
Belarus          3
Estonia          3
Latvia           3
Lithuania        3
Russia           0
Ukraine          3

✅ Saved country_clusters.csv


In [10]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

feature_cols = [
    'log_gdp',          # economic capacity
    'health_exp',       # healthcare investment
    'pop_density',      # transmission risk
    'log_cons_total',   # overall antibiotic pressure
    'log_cons_per_class', # class-specific AMR pressure
    'sanitation',       # infection prevention
    'water',            # infection prevention
    'hospital_beds',    # healthcare capacity
    'mean_temp',        
    'aware_access',     
]

# Domain-justified weights:
# Higher weight = more AMR-relevant signal
# Lower weight = infrastructure proxies that distort KZ placement
feature_weights = {
    'log_gdp':            1.0,
    'health_exp':         1.2,   # was 1.5
    'pop_density':        0.9,   # was 0.8
    'log_cons_total':     1.5,   # was 2.0
    'log_cons_per_class': 1.5,   # was 2.0
    'sanitation':         0.8,   # was 0.7
    'water':              0.8,   # was 0.5  ← too aggressive before
    'hospital_beds':      0.8,   # was 0.5  ← too aggressive before
    'mean_temp':          0.9,   # was 0.8
    'aware_access':       1.2,   # was 1.5
}
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(country_profile[feature_cols])

# Apply weights after scaling so they're on the same scale
weights   = np.array([feature_weights[f] for f in feature_cols])
X_cluster = X_scaled * weights

kmeans = KMeans(n_clusters=6, random_state=42, n_init=20)
country_profile['cluster'] = kmeans.fit_predict(X_cluster)

kz_cluster = country_profile.loc['Kazakhstan', 'cluster']
print(f"Kazakhstan is in cluster {kz_cluster}")

print("\nCountries in Kazakhstan's cluster:")
cluster_members = country_profile[country_profile['cluster'] == kz_cluster].drop(columns='cluster')
print(cluster_members.index.tolist())

print("\nPost-Soviet cluster assignments:")
check = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
ps_df = country_profile[country_profile.index.isin(check)][['cluster']]
print(ps_df.to_string())

# ── Distance to centroid using weighted space ─────────────────────────────────
centroid   = kmeans.cluster_centers_[kz_cluster]
distances  = {}
for country in cluster_members.index:
    row    = country_profile.loc[country, feature_cols].values.reshape(1, -1)
    scaled = scaler.transform(row) * weights
    distances[country] = np.linalg.norm(scaled - centroid)

dist_df = pd.Series(distances).sort_values().rename('distance_to_centroid')
print("\nDistance to cluster centroid (lower = more similar):")
print(dist_df.to_string())

country_profile.to_csv('/kaggle/working/country_clusters.csv')
print("\n✅ Saved country_clusters.csv")

Kazakhstan is in cluster 3

Countries in Kazakhstan's cluster:
['Qatar', 'Kazakhstan']

Post-Soviet cluster assignments:
           cluster
Belarus          0
Estonia          1
Latvia           1
Lithuania        1
Russia           0
Ukraine          0

Distance to cluster centroid (lower = more similar):
Qatar         3.24776
Kazakhstan    3.24776

✅ Saved country_clusters.csv


In [17]:
# =============================================================================
# LOCO — Reference Country Groups for Kazakhstan Zero-Shot Validation
# =============================================================================
import gc
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

TEMPORAL_CUTOFF = 2020
ES_VAL_YEARS    = [2019, 2020]

# ── Reference groups ──────────────────────────────────────────────────────────
post_soviet      = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
structural_peers = ['Qatar', 'Turkey', 'Romania', 'Bulgaria', 'Hungary', 'Poland']
all_except_kz    = [c for c in full_df['country'].unique() if c != 'Kazakhstan']

loco_groups = {
    'post_soviet':      post_soviet,
    'structural_peers': structural_peers,
    'all_except_kz':    all_except_kz,
}
EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']
source_df = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
y_source  = source_df['resistance'].values
SCALE_POS_WEIGHT = (y_source == 0).sum() / (y_source == 1).sum()
print(f"SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT:.4f}")
del y_source; gc.collect()

CATBOOST_PARAMS = {
    'learning_rate':         0.03,
    'depth':                 7,
    'l2_leaf_reg':           3,
    'eval_metric':           'AUC',
    'random_seed':           42,
    'verbose':               200,
    'early_stopping_rounds': 100,
    'iterations':            5000,
    'task_type':             'GPU',
    'scale_pos_weight':      SCALE_POS_WEIGHT,
}

def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def make_kz_pool(df):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    return Pool(data=X, cat_features=CAT_FEATURES)
loco_results = {}

for group_name, countries in loco_groups.items():
    print(f"\n{'='*60}")
    print(f"[{group_name}] — {len(countries)} countries")

    available = [c for c in countries if c in full_df['country'].unique()]
    missing   = [c for c in countries if c not in full_df['country'].unique()]
    if missing:
        print(f"  ⚠️  Not in dataset: {missing}")

    train_source = full_df[full_df['country'].isin(available)].copy()

    if len(train_source) == 0:
        print(f"  Skipped — no training data")
        continue

    # ── Per-fold early stopping: ES val = 2019-2020, ES train = ≤2018 ─────────
    es_train_df = train_source[
        (train_source['year'] <= TEMPORAL_CUTOFF) &
        (~train_source['year'].isin(ES_VAL_YEARS))
    ].copy()
    es_val_df = train_source[
        train_source['year'].isin(ES_VAL_YEARS)
    ].copy()

    if len(es_val_df) < 200:
        print(f"  Skipped — ES val too small ({len(es_val_df)} rows)")
        continue

    print(f"  ES train : {len(es_train_df):,} rows")
    print(f"  ES val   : {len(es_val_df):,} rows")
    print(f"  KZ input : {len(kz_final):,} rows")

    # ── Stage 1: calibrate iterations ─────────────────────────────────────────
    cal_model = CatBoostClassifier(**CATBOOST_PARAMS)
    cal_model.fit(make_pool(es_train_df), eval_set=make_pool(es_val_df))
    best_iter = cal_model.best_iteration_
    print(f"  Best iteration: {best_iter}")
    del cal_model; gc.collect()

    # ── Stage 2: retrain on full ≤2020 data from this group
    final_train_df = train_source[train_source['year'] <= TEMPORAL_CUTOFF].copy()
    final_params   = {
        **CATBOOST_PARAMS,
        'iterations':            best_iter,
        'early_stopping_rounds': None,
        'verbose':               100,
    }

    model = CatBoostClassifier(**final_params)
    model.fit(make_pool(final_train_df))
    del final_train_df, train_source; gc.collect()

    # ── Predict on KZ 
    kz_pool      = make_kz_pool(kz_final)
    kz_proba     = model.predict_proba(kz_pool)[:, 1]
    kz_pred_bin  = (kz_proba >= 0.5).astype(int)

    # ── Evaluate on ES val set (proxy for generalization quality)
    val_proba = model.predict_proba(make_pool(es_val_df))[:, 1]
    y_val     = es_val_df['resistance'].values
    val_auc   = roc_auc_score(y_val, val_proba)   if y_val.std() > 0 else np.nan
    val_auprc = average_precision_score(y_val, val_proba) if y_val.std() > 0 else np.nan
    val_brier = brier_score_loss(y_val, val_proba)

    print(f"  Val AUC={val_auc:.4f} | AUPRC={val_auprc:.4f} | Brier={val_brier:.4f}")
    print(f"  KZ pred dist: 0={int((kz_pred_bin==0).sum())} | 1={int((kz_pred_bin==1).sum())}")
    print(f"  KZ mean prob: {kz_proba.mean():.4f}")

    loco_results[group_name] = {
        'model':        model,
        'kz_proba':     kz_proba,
        'kz_pred_bin':  kz_pred_bin,
        'best_iter':    best_iter,
        'n_train':      len(es_train_df) + len(es_val_df),
        'n_countries':  len(available),
        'val_auc':      val_auc,
        'val_auprc':    val_auprc,
        'val_brier':    val_brier,
    }

    del model, es_train_df, es_val_df; gc.collect()

#Attach predictions to kz_final
print("\n── LOCO Prediction Comparison ──────────────────────────────────────")

for group_name, res in loco_results.items():
    kz_final[f'prob_{group_name}']  = res['kz_proba']
    kz_final[f'pred_{group_name}']  = res['kz_pred_bin']

# Agreement between post_soviet and all_except_kz
if 'post_soviet' in loco_results and 'all_except_kz' in loco_results:
    agreement = (kz_final['pred_post_soviet'] == kz_final['pred_all_except_kz']).mean()
    print(f"Agreement (post_soviet vs all_except_kz): {agreement:.1%}")

#average probability across groups
prob_cols = [f'prob_{g}' for g in loco_results]
pred_cols = [f'pred_{g}' for g in loco_results]

kz_final['prob_loco_ensemble'] = kz_final[prob_cols].mean(axis=1)
kz_final['pred_loco_ensemble'] = (kz_final['prob_loco_ensemble'] >= 0.5).astype(int)

# Majority vote as secondary ensemble
kz_final['pred_loco_majority'] = (kz_final[pred_cols].sum(axis=1) > len(pred_cols) / 2).astype(int)

print("\nLOCO ensemble (avg prob) prediction distribution:")
print(kz_final['pred_loco_ensemble'].value_counts())
print("\nLOCO majority vote distribution:")
print(kz_final['pred_loco_majority'].value_counts())

summary_rows = []
for group_name, res in loco_results.items():
    summary_rows.append({
        'group':       group_name,
        'n_countries': res['n_countries'],
        'n_train':     res['n_train'],
        'best_iter':   res['best_iter'],
        'val_AUC':     round(res['val_auc'],  4),
        'val_AUPRC':   round(res['val_auprc'],4),
        'val_Brier':   round(res['val_brier'],4),
        'kz_mean_prob':round(res['kz_proba'].mean(), 4),
        'kz_pct_resistant': round(res['kz_pred_bin'].mean() * 100, 1),
    })

summary_df = pd.DataFrame(summary_rows)
print("\n── LOCO Group Summary ───────────────────────────────────────────────")
print(summary_df.to_string(index=False))
summary_df.to_csv('/kaggle/working/loco_group_summary.csv', index=False)

#Calibration plot for each group
# Uses ES val set as proxy calibration reference
fig, axes = plt.subplots(1, len(loco_results), figsize=(6*len(loco_results), 5))
if len(loco_results) == 1:
    axes = [axes]

for ax, (group_name, res) in zip(axes, loco_results.items()):
    ax.set_title(f'{group_name}\nVal AUC={res["val_auc"]:.4f}')
    ax.set_xlabel('Mean predicted prob')
    ax.set_ylabel('KZ mean resistance prob')
    ax.hist(res['kz_proba'], bins=30, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(res['kz_proba'].mean(), color='red', linestyle='--',
               label=f'Mean={res["kz_proba"].mean():.3f}')
    ax.legend()

plt.suptitle('KZ Predicted Probability Distribution by LOCO Group', fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/loco_kz_prob_dist.png', dpi=150, bbox_inches='tight')
plt.show()

kz_final.to_csv('/kaggle/working/kz_final_with_loco_preds.csv', index=False)
print("\n✅ Saved: loco_group_summary.csv + kz_final_with_loco_preds.csv + loco_kz_prob_dist.png")

SCALE_POS_WEIGHT : 4.5446

[post_soviet] — 6 countries
  ES train : 282,170 rows
  ES val   : 163,657 rows
  KZ input : 183 rows


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8604288	best: 0.8604288 (0)	total: 66.7ms	remaining: 5m 33s
200:	test: 0.8949423	best: 0.8949466 (199)	total: 6.46s	remaining: 2m 34s
400:	test: 0.8962906	best: 0.8962906 (400)	total: 12.8s	remaining: 2m 26s
600:	test: 0.8967609	best: 0.8967609 (600)	total: 19.1s	remaining: 2m 19s
800:	test: 0.8969373	best: 0.8969518 (799)	total: 25.3s	remaining: 2m 12s
bestTest = 0.8969518244
bestIteration = 799
Shrink model to first 800 iterations.
  Best iteration: 799


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 31.7ms	remaining: 25.3s
100:	total: 3.44s	remaining: 23.8s
200:	total: 6.98s	remaining: 20.8s
300:	total: 10.5s	remaining: 17.3s
400:	total: 14s	remaining: 13.9s
500:	total: 17.5s	remaining: 10.4s
600:	total: 21s	remaining: 6.92s
700:	total: 24.5s	remaining: 3.42s
798:	total: 28s	remaining: 0us


KeyError: "['organism'] not in index"

## 11. Zero-Shot Prediction for Kazakhstan

Model trained exclusively on non-Kazakh global data, applied to Kazakhstan using macroeconomic proxy features.
A prediction grid crosses Kazakhstan antibiotic-year rows with 6 target pathogen species.

**Encoding:** Fit on source training data only. KZ antibiotic names normalized to lowercase to ensure
vocabulary alignment. Unknown labels mapped to -1.


In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, recall_score, classification_report
)

print("=== Kazakhstan Zero-Shot Prediction (CatBoost) ===")

# ── Load data ─────────────────────────────────────────────────────────────────
full_df  = pd.read_csv('/kaggle/input/datasets/user/full-df/full_amr_with_macro.csv', low_memory=False)
kz_final = pd.read_csv('/kaggle/input/datasets/user/kzzzzz/kazakhstan_final_prediction_input_2017_2023.csv', low_memory=False)

# ── Configuration ─────────────────────────────────────────────────────────────
EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']
TEMPORAL_CUTOFF   = 2020

TARGET_ORGANISMS = [
    'Escherichia coli', 'Klebsiella pneumoniae', 'Staphylococcus aureus',
    'Pseudomonas aeruginosa', 'Acinetobacter baumannii', 'Enterococcus faecium'
]
ANTIBIOTIC_TO_CLASS_MAP = {
    # Carbapenems
    'Meropenem': 'Carbapenems', 'Imipenem': 'Carbapenems',
    'Doripenem': 'Carbapenems', 'Ertapenem': 'Carbapenems',
    'Meropenem vaborbactam': 'Carbapenems',
    'Meropenem/ Vaborbactam at 8': 'Carbapenems',
    'Imipenem/ Relebactam': 'Carbapenems',
    # Fluoroquinolones
    'Ciprofloxacin': 'Fluoroquinolones', 'Levofloxacin': 'Fluoroquinolones',
    'Moxifloxacin': 'Fluoroquinolones',  'Ofloxacin': 'Fluoroquinolones',
    'Norfloxacin': 'Fluoroquinolones',
    # Cephalosporins
    'Ceftazidime': 'Cephalosporins',  'Cefepime': 'Cephalosporins',
    'Ceftriaxone': 'Cephalosporins',  'Cefixime': 'Cephalosporins',
    'Cefazolin': 'Cephalosporins',    'Cefotaxime': 'Cephalosporins',
    'Cefuroxime': 'Cephalosporins',   'Cefpodoxime': 'Cephalosporins',
    'Cefoxitin': 'Cephalosporins',    'Ceftolozane': 'Cephalosporins',
    'Ceftaroline': 'Cephalosporins',  'Ceftazidime avibactam': 'Cephalosporins',
    'Ceftolozane tazobactam': 'Cephalosporins', 'Cefiderocol': 'Cephalosporins',
    'Ceftazidime/ Avibactam': 'Cephalosporins', 'Ceftolozane/ Tazobactam': 'Cephalosporins',
    'Ceftibuten': 'Cephalosporins',
    # Aminoglycosides
    'Amikacin': 'Aminoglycosides',  'Gentamicin': 'Aminoglycosides',
    'Kanamycin': 'Aminoglycosides', 'Tobramycin': 'Aminoglycosides',
    'Netilmicin': 'Aminoglycosides',
    # Penicillins
    'Ampicillin': 'Penicillins',
    'Amoxicillin- clavulanic acid': 'Penicillins',
    'Amoxycillin clavulanate': 'Penicillins',
    'Amoxicillin-\nclavulanic acid': 'Penicillins',
    'Penicillin': 'Penicillins',
    'Piperacillin': 'Penicillins',
    'Piperacillin-tazobactam': 'Penicillins',
    'Piperacillin tazobactam': 'Penicillins',
    'Piperacillin-\ntazobactam': 'Penicillins',
    'Oxacillin': 'Penicillins',
    'Ampicillin sulbactam': 'Penicillins',
    'Ampicillin/ Sulbactam': 'Penicillins',
    # Macrolides
    'Azithromycin': 'Macrolides', 'Clarithromycin': 'Macrolides',
    'Erythromycin': 'Macrolides', 'Midecamycin': 'Macrolides',
    'Roxithromycin': 'Macrolides',
    # Glycopeptides
    'Vancomycin': 'Glycopeptides', 'Teicoplanin': 'Glycopeptides',
    # Tetracyclines
    'Doxycycline': 'Tetracyclines',  'Tetracycline': 'Tetracyclines',
    'Minocycline': 'Tetracyclines',  'Tigecycline': 'Tetracyclines',
    'Omadacycline': 'Tetracyclines',
    # Oxazolidinones
    'Linezolid': 'Oxazolidinones',
    # Polymyxins
    'Colistin': 'Polymyxins', 'Polymyxin B': 'Polymyxins',
    # Sulfonamides
    'Trimethoprim / sulfamethoxazole': 'Sulfonamides',
    'Trimethoprim-sulfamethoxazole':   'Sulfonamides',
    'Trimethoprim/ Sulfamethoxazole':  'Sulfonamides',
    'Trimethoprim sulfa':              'Sulfonamides',
    'Trimethoprim':                    'Sulfonamides',
    'Sulfasodimidine':                 'Sulfonamides',
    # Nitroimidazoles
    'Metronidazole': 'Nitroimidazoles',
    # Phosphonic acids
    'Fosfomycin': 'Phosphonic acids',
    # Amphenicols
    'Chloramphenicol': 'Amphenicols',
    'Thiamphenicol':   'Amphenicols',
    # Monobactams
    'Aztreonam': 'Monobactams', 'Aztreonam/ Avibactam': 'Monobactams',
    # Lincosamides
    'Clindamycin': 'Lincosamides',
    # Lipopeptides
    'Daptomycin': 'Lipopeptides',
    # Streptogramins
    'Quinupristin dalfopristin': 'Streptogramins',
    # Nitrofurans (regional KZ drugs)
    'Furazidin':     'Nitrofurans',
    'Nitrofurantoin': 'Nitrofurans',
    # Quinolines (regional KZ drugs)
    'Nitroxoline': 'Quinolines',
}
#lowercase variants
ANTIBIOTIC_TO_CLASS_MAP.update({
    k.lower(): v for k, v in list(ANTIBIOTIC_TO_CLASS_MAP.items())
})

MODEL_PATH = '/kaggle/working/catboost_retrained.cbm'
print(f"Loading model from {MODEL_PATH} ...")
model_final = CatBoostClassifier()
model_final.load_model(MODEL_PATH)
print(f"  Loaded — tree count: {model_final.tree_count_}")

def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def print_metrics(y_true, y_prob, label=''):
    y_pred = (y_prob >= 0.5).astype(int)
    auc   = roc_auc_score(y_true, y_prob)           if y_true.std() > 0 else np.nan
    auprc = average_precision_score(y_true, y_prob) if y_true.std() > 0 else np.nan
    f1    = f1_score(y_true, y_pred,     zero_division=0)
    rec   = recall_score(y_true, y_pred, zero_division=0)
    prefix = f"[{label}] " if label else ""
    print(f"\n{prefix}AUC    : {auc:.4f}")
    print(f"{prefix}AUPRC  : {auprc:.4f}")
    print(f"{prefix}F1     : {f1:.4f}")
    print(f"{prefix}Recall : {rec:.4f}")
    print(f"\n{prefix}Classification report:")
    print(classification_report(y_true, y_pred,
                                target_names=['Susceptible', 'Resistant'],
                                zero_division=0))
    return {'auc': auc, 'auprc': auprc, 'f1': f1, 'recall': rec}


kz_df = kz_final.copy()

prediction_rows = []
for _, row in kz_df.iterrows():
    for org in TARGET_ORGANISMS:
        new_row = row.to_dict()
        new_row['organism'] = org
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
kz_df = kz_df.loc[:, ~kz_df.columns.duplicated()]
kz_df['country'] = 'Kazakhstan'
print(f"KZ rows after organism expansion: {len(kz_df):,}")

# ── Antibiotic harmonization
kz_df['antibiotic'] = (kz_df['antibiotic']
                       .astype(str).str.strip().str.lower()
                       .str.replace('/', ' / ', regex=False)
                       .str.replace(r'\s+', ' ', regex=True)
                       .str.strip())

antibiotic_harmonize = {
    'amoxicillin / clavulanic acid':            'amoxicillin- clavulanic acid',
    'amoxicillin and beta-lactamase inhibitor': 'amoxicillin- clavulanic acid',
    'amoxicillin':                              'amoxycillin clavulanate',
    'imipenem / cilastatin':                    'imipenem',
    'sulfamethoxazole / trimethoprim':          'trimethoprim / sulfamethoxazole',
    'sulfamethoxazole and trimethoprim':        'trimethoprim / sulfamethoxazole',
    'vancomycin (iv)':                          'vancomycin',
    'fosfomycin (iv)':                          'fosfomycin',
    'benzylpenicillin':                         'penicillin',
    'cefpodoxime proxetil':                     'cefpodoxime',
}
kz_df['antibiotic'] = kz_df['antibiotic'].replace(antibiotic_harmonize)
print(f"KZ unique antibiotics after harmonization: {kz_df['antibiotic'].nunique()}")

# ── Casing fix: map to source vocabulary casing
source_df          = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
source_antibiotics = set(source_df['antibiotic'].unique())
source_lower_map   = {ab.lower().strip(): ab for ab in source_antibiotics}

kz_df['antibiotic'] = kz_df['antibiotic'].apply(
    lambda x: source_lower_map.get(x.lower().strip(), x)
)

# ── Vocabulary match diagnostic
kz_antibiotics_final = set(kz_df['antibiotic'].unique())
matched_final   = kz_antibiotics_final & source_antibiotics
unmatched_final = kz_antibiotics_final - source_antibiotics
print(f"\nVocabulary match: {len(matched_final)}/{len(kz_antibiotics_final)}")
if unmatched_final:
    print(f"Unmatched (regional drugs — CatBoost will treat as unseen): {sorted(unmatched_final)}")
else:
    print("✅ All antibiotics matched to source vocabulary")

kz_df['antibiotic_class'] = kz_df['antibiotic'].map(ANTIBIOTIC_TO_CLASS_MAP)
kz_df['antibiotic_class'] = kz_df.apply(
    lambda row: row['antibiotic_class']
    if pd.notna(row['antibiotic_class'])
    else ANTIBIOTIC_TO_CLASS_MAP.get(row['antibiotic'].lower().strip(), np.nan),
    axis=1
)

unmapped_class = kz_df[kz_df['antibiotic_class'].isna()]['antibiotic'].value_counts()
coverage = kz_df['antibiotic_class'].notna().mean() * 100
print(f"\nantibiotic_class coverage: {coverage:.1f}%")
if len(unmapped_class) > 0:
    print(f"Still unmapped: {unmapped_class.index.tolist()}")
else:
    print("✅ All antibiotics mapped to class")

#NaN check across
print("\nFeature NaN check:")
for f in FEATURES:
    if f in kz_df.columns:
        pct = kz_df[f].isna().mean() * 100
        if pct > 0:
            print(f"  ⚠ {f}: {pct:.1f}% null")
    else:
        print(f"  ✗ {f}: MISSING from kz_df")

#Label check
has_kz_labels = ('resistance' in kz_df.columns) and (kz_df['resistance'].notna().sum() > 0)
if has_kz_labels:
    print(f"KZ resistance prevalence: {kz_df['resistance'].mean():.1%}")
else:
    print("No KZ resistance labels — prediction only (zero-shot)")

#Predict
kz_pool   = make_pool(kz_df, has_labels=has_kz_labels)
y_kz_prob = model_final.predict_proba(kz_pool)[:, 1]


kz_metrics = {}
if has_kz_labels:
    y_kz = kz_df['resistance'].values
    kz_metrics = print_metrics(y_kz, y_kz_prob, label='KZ Zero-Shot')

wanted_cols = ['country', 'year', 'organism', 'antibiotic', 'antibiotic_class']
if has_kz_labels:
    wanted_cols += ['resistance']
save_cols = [c for c in wanted_cols if c in kz_df.columns]
missing   = [c for c in wanted_cols if c not in kz_df.columns]
if missing:
    print(f"  Note: columns not in KZ data (skipped): {missing}")

kz_pred_df = kz_df[save_cols].copy()
kz_pred_df['pred_proba'] = y_kz_prob
kz_pred_df['pred_label'] = (y_kz_prob >= 0.5).astype(int)

kz_pred_df.to_csv('/kaggle/working/kz_predictions.csv', index=False)
print(f"\n✅ Saved kz_predictions.csv | {len(kz_pred_df):,} rows")

#Per antibiotic-class breakdown
print("\nPer-class predicted resistance rates:")

def class_summary_row(g):
    row = {'n': len(g), 'pred_resistance': g['pred_proba'].mean()}
    if has_kz_labels and g['resistance'].std() > 0:
        y_true = g['resistance'].values
        y_prob = g['pred_proba'].values
        y_pred = (y_prob >= 0.5).astype(int)
        row['true_resistance'] = y_true.mean()
        row['AUC']    = roc_auc_score(y_true, y_prob)
        row['F1']     = f1_score(y_true, y_pred, zero_division=0)
        row['Recall'] = recall_score(y_true, y_pred, zero_division=0)
    return pd.Series(row)

if 'antibiotic_class' in kz_pred_df.columns:
    class_summary = (
        kz_pred_df.groupby('antibiotic_class')
        .apply(class_summary_row)
        .reset_index()
        .sort_values('pred_resistance', ascending=False)
    )
    print(class_summary.to_string(index=False))
    class_summary.to_csv('/kaggle/working/kz_class_summary.csv', index=False)
    print("✅ Saved kz_class_summary.csv")
else:
    print("  'antibiotic_class' not in KZ data — skipping class breakdown")

print("\nPer-organism predicted resistance rates:")

def org_summary_row(g):
    row = {'n': len(g), 'pred_resistance': g['pred_proba'].mean()}
    if has_kz_labels and g['resistance'].std() > 0:
        y_true = g['resistance'].values
        y_prob = g['pred_proba'].values
        y_pred = (y_prob >= 0.5).astype(int)
        row['true_resistance'] = y_true.mean()
        row['F1']     = f1_score(y_true, y_pred, zero_division=0)
        row['Recall'] = recall_score(y_true, y_pred, zero_division=0)
    return pd.Series(row)

org_summary = (
    kz_pred_df.groupby('organism')
    .apply(org_summary_row)
    .reset_index()
    .sort_values('pred_resistance', ascending=False)
)
print(org_summary.to_string(index=False))
org_summary.to_csv('/kaggle/working/kz_organism_summary.csv', index=False)
print("✅ Saved kz_organism_summary.csv")



print("\nYear × Organism predicted resistance rates:")

kz_pred_df['pred_proba'] = y_kz_prob 

year_org_summary = (
    kz_pred_df.groupby(['year', 'organism'])['pred_proba']
    .mean()
    .reset_index()
    .rename(columns={'pred_proba': 'pred_resistance'})
    .sort_values(['year', 'pred_resistance'], ascending=[True, False])
)
print(year_org_summary.to_string(index=False))
year_org_summary.to_csv('/kaggle/working/kz_year_organism_summary.csv', index=False)
print("✅ Saved kz_year_organism_summary.csv")

print("\nYear × Antibiotic class predicted resistance rates:")

year_class_summary = (
    kz_pred_df.groupby(['year', 'antibiotic_class'])['pred_proba']
    .mean()
    .reset_index()
    .rename(columns={'pred_proba': 'pred_resistance'})
    .sort_values(['year', 'pred_resistance'], ascending=[True, False])
)
print(year_class_summary.to_string(index=False))
year_class_summary.to_csv('/kaggle/working/kz_year_class_summary.csv', index=False)
print("✅ Saved kz_year_class_summary.csv")

print("\n2021-2023 predictions (for concordance against ground truth):")

recent_org = (
    kz_pred_df[kz_pred_df['year'] >= 2021]
    .groupby('organism')['pred_proba']
    .mean()
    .reset_index()
    .rename(columns={'pred_proba': 'pred_resistance_2021_2023'})
    .sort_values('pred_resistance_2021_2023', ascending=False)
)
print(recent_org.to_string(index=False))

recent_class = (
    kz_pred_df[kz_pred_df['year'] >= 2021]
    .groupby('antibiotic_class')['pred_proba']
    .mean()
    .reset_index()
    .rename(columns={'pred_proba': 'pred_resistance_2021_2023'})
    .sort_values('pred_resistance_2021_2023', ascending=False)
)
print(recent_class.to_string(index=False))

recent_org.to_csv('/kaggle/working/kz_2021_2023_organism.csv', index=False)
recent_class.to_csv('/kaggle/working/kz_2021_2023_class.csv', index=False)
print("✅ Saved kz_2021_2023_organism.csv and kz_2021_2023_class.csv")


print("\nTrend: year-over-year mean predicted resistance (all organisms):")

yearly_trend = (
    kz_pred_df.groupby('year')['pred_proba']
    .mean()
    .reset_index()
    .rename(columns={'pred_proba': 'mean_pred_resistance'})
)
yearly_trend['yoy_change'] = yearly_trend['mean_pred_resistance'].diff().round(4)
print(yearly_trend.to_string(index=False))
yearly_trend.to_csv('/kaggle/working/kz_yearly_trend.csv', index=False)
print("✅ Saved kz_yearly_trend.csv")


print(f"\n{'='*50}")
print(f"Final model summary:")
print(f"  Model path    : {MODEL_PATH}")
print(f"  Tree count    : {model_final.tree_count_}")
print(f"  KZ predictions: {len(kz_pred_df):,}")
if kz_metrics:
    print(f"  KZ AUC        : {kz_metrics['auc']:.4f}")
    print(f"  KZ AUPRC      : {kz_metrics['auprc']:.4f}")
    print(f"  KZ F1         : {kz_metrics['f1']:.4f}")
    print(f"  KZ Recall     : {kz_metrics['recall']:.4f}")

In [ ]:
import gc
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import matplotlib.pyplot as plt

print("=== LOCO Validation for Kazakhstan (Fixed Version) ===\n")

# ----------------------------- CONFIG -----------------------------
TEMPORAL_CUTOFF = 2020
ES_VAL_YEARS    = [2019, 2020]

# Reference groups
post_soviet      = ['Russia', 'Ukraine', 'Belarus', 'Lithuania', 'Latvia', 'Estonia']
structural_peers = ['Qatar', 'Turkey', 'Romania', 'Bulgaria', 'Hungary', 'Poland']
all_except_kz    = [c for c in full_df['country'].unique() if c != 'Kazakhstan']

loco_groups = {
    'post_soviet':      post_soviet,
    'structural_peers': structural_peers,
    'all_except_kz':    all_except_kz,
}

EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']

# ----------------------------- LOAD DATA -----------------------------
full_df = pd.read_csv('/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv', low_memory=False)
kz_final = pd.read_csv('/kaggle/working/kazakhstan_final_prediction_input_2017_2023.csv')

print(f"Full dataset : {len(full_df):,} rows")
print(f"KZ input     : {len(kz_final):,} rows\n")

# ----------------------------- FIX KZ DATAFRAME -----------------------------
# Add missing categorical columns required by the model
kz_final['country'] = 'Kazakhstan'
kz_final['data_source'] = 'KZ_DID'

# Expand with target organisms (critical step!)
TARGET_ORGANISMS = [
    'Escherichia coli', 'Klebsiella pneumoniae', 'Staphylococcus aureus',
    'Pseudomonas aeruginosa', 'Acinetobacter baumannii', 'Enterococcus faecium'
]

prediction_rows = []
for _, row in kz_final.iterrows():
    for org in TARGET_ORGANISMS:
        new_row = row.to_dict()
        new_row['organism'] = org
        prediction_rows.append(new_row)

kz_df = pd.DataFrame(prediction_rows)
print(f"KZ rows after organism expansion: {len(kz_df):,}\n")

# ----------------------------- SCALE POS WEIGHT -----------------------------
source_df = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
SCALE_POS_WEIGHT = (source_df['resistance'] == 0).sum() / (source_df['resistance'] == 1).sum()
print(f"SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT:.4f}\n")

# ----------------------------- CATBOOST PARAMS -----------------------------
CATBOOST_PARAMS = {
    'learning_rate':         0.03,
    'depth':                 7,
    'l2_leaf_reg':           3,
    'eval_metric':           'AUC',
    'random_seed':           42,
    'verbose':               200,
    'early_stopping_rounds': 100,
    'iterations':            5000,
    'task_type':             'GPU',
    'scale_pos_weight':      SCALE_POS_WEIGHT,
}

# ----------------------------- POOL HELPERS (ROBUST) -----------------------------
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

def make_kz_pool(df):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    return Pool(data=X, cat_features=CAT_FEATURES)

# ----------------------------- MAIN LOCO LOOP -----------------------------
loco_results = {}

for group_name, countries in loco_groups.items():
    print(f"{'='*70}")
    print(f"[{group_name.upper()}] — {len(countries)} countries")
    
    available = [c for c in countries if c in full_df['country'].unique()]
    if not available:
        print("  Skipped — no data available")
        continue
        
    train_source = full_df[full_df['country'].isin(available)].copy()
    
    # Early stopping split
    es_train_df = train_source[
        (train_source['year'] <= TEMPORAL_CUTOFF) & 
        (~train_source['year'].isin(ES_VAL_YEARS))
    ].copy()
    
    es_val_df = train_source[train_source['year'].isin(ES_VAL_YEARS)].copy()
    
    if len(es_val_df) < 200:
        print(f"  Skipped — ES val too small ({len(es_val_df)} rows)")
        continue
    
    print(f"  ES train : {len(es_train_df):,} rows")
    print(f"  ES val   : {len(es_val_df):,} rows")
    
    # Stage 1: Find best iterations
    cal_model = CatBoostClassifier(**CATBOOST_PARAMS)
    cal_model.fit(make_pool(es_train_df), eval_set=make_pool(es_val_df))
    best_iter = cal_model.best_iteration_
    print(f"  Best iteration: {best_iter}")
    del cal_model; gc.collect()
    
    # Stage 2: Final model on all ≤2020 data
    final_train_df = train_source[train_source['year'] <= TEMPORAL_CUTOFF].copy()
    final_params = {**CATBOOST_PARAMS, 
                    'iterations': best_iter, 
                    'early_stopping_rounds': None, 
                    'verbose': 100}
    
    model = CatBoostClassifier(**final_params)
    model.fit(make_pool(final_train_df))
    del final_train_df; gc.collect()
    
    # Predict on Kazakhstan
    kz_proba = model.predict_proba(make_kz_pool(kz_df))[:, 1]
    kz_pred_bin = (kz_proba >= 0.5).astype(int)
    
    # Evaluate on ES val set
    val_proba = model.predict_proba(make_pool(es_val_df))[:, 1]
    y_val = es_val_df['resistance'].values
    val_auc   = roc_auc_score(y_val, val_proba)
    val_auprc = average_precision_score(y_val, val_proba)
    val_brier = brier_score_loss(y_val, val_proba)
    
    print(f"  Val AUC={val_auc:.4f} | AUPRC={val_auprc:.4f} | Brier={val_brier:.4f}")
    print(f"  KZ mean prob = {kz_proba.mean():.4f} | Resistant = {kz_pred_bin.mean():.1%}\n")
    
    loco_results[group_name] = {
        'model': model,
        'kz_proba': kz_proba,
        'kz_pred_bin': kz_pred_bin,
        'best_iter': best_iter,
        'val_auc': val_auc,
        'val_auprc': val_auprc,
        'val_brier': val_brier,
    }
    
    del model, es_train_df, es_val_df
    gc.collect()

# ----------------------------- ENSEMBLE & SUMMARY -----------------------------
print("="*70)
print("LOCO ENSEMBLE RESULTS")

for group_name, res in loco_results.items():
    kz_df[f'prob_{group_name}'] = res['kz_proba']
    kz_df[f'pred_{group_name}'] = res['kz_pred_bin']

prob_cols = [f'prob_{g}' for g in loco_results]
kz_df['prob_loco_ensemble'] = kz_df[prob_cols].mean(axis=1)
kz_df['pred_loco_ensemble'] = (kz_df['prob_loco_ensemble'] >= 0.5).astype(int)

print("\nFinal Ensemble Prediction Distribution:")
print(kz_df['pred_loco_ensemble'].value_counts(normalize=True))

# Summary table
summary = []
for g, r in loco_results.items():
    summary.append({
        'Group': g,
        'Countries': r['n_countries'] if 'n_countries' in r else len(loco_groups[g]),
        'Val_AUC': round(r['val_auc'], 4),
        'Val_AUPRC': round(r['val_auprc'], 4),
        'KZ_Mean_Prob': round(r['kz_proba'].mean(), 4),
        'KZ_%_Resistant': round(r['kz_pred_bin'].mean()*100, 1)
    })

summary_df = pd.DataFrame(summary)
print("\n", summary_df)

# Save
kz_df.to_csv('/kaggle/working/kz_final_with_loco_preds.csv', index=False)
summary_df.to_csv('/kaggle/working/loco_group_summary.csv', index=False)

print("\n✅ LOCO completed successfully!")
print("Files saved:")
print("   • kz_final_with_loco_preds.csv")
print("   • loco_group_summary.csv")

=== LOCO Validation for Kazakhstan (Fixed Version) ===

Full dataset : 20,446,868 rows
KZ input     : 183 rows

KZ rows after organism expansion: 1,098

SCALE_POS_WEIGHT : 4.5446

[POST_SOVIET] — 6 countries
  ES train : 282,170 rows
  ES val   : 163,657 rows


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8576874	best: 0.8576874 (0)	total: 61.1ms	remaining: 5m 5s
200:	test: 0.8996046	best: 0.8997762 (187)	total: 7.22s	remaining: 2m 52s
bestTest = 0.8997761607
bestIteration = 187
Shrink model to first 188 iterations.
  Best iteration: 187


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 13.1ms	remaining: 2.44s
100:	total: 1.24s	remaining: 1.05s
186:	total: 2.13s	remaining: 0us
  Val AUC=0.9007 | AUPRC=0.8071 | Brier=0.1728
  KZ mean prob = 0.2293 | Resistant = 1.0%

[STRUCTURAL_PEERS] — 6 countries
  ES train : 758,387 rows
  ES val   : 315,152 rows


...

0:	total: 127ms	remaining: 45.6s
100:	total: 10.7s	remaining: 27.4s
200:	total: 20.5s	remaining: 16.2s
300:	total: 30.4s	remaining: 5.95s
359:	total: 36.3s	remaining: 0us
  Val AUC=0.9104 | AUPRC=0.7852 | Brier=0.1516
  KZ mean prob = 0.5459 | Resistant = 66.7%

[ALL_EXCEPT_KZ] — 85 countries
  ES train : 11,207,215 rows
  ES val   : 3,607,997 rows


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8676571	best: 0.8676571 (0)	total: 1.94s	remaining: 2h 41m 48s
200:	test: 0.8905031	best: 0.8905031 (200)	total: 5m 24s	remaining: 2h 8m 56s
400:	test: 0.8920424	best: 0.8920424 (400)	total: 9m 58s	remaining: 1h 54m 29s
600:	test: 0.8924293	best: 0.8924572 (590)	total: 14m 4s	remaining: 1h 42m 59s
800:	test: 0.8924772	best: 0.8924980 (784)	total: 18m	remaining: 1h 34m 25s
1000:	test: 0.8926205	best: 0.8926205 (1000)	total: 21m 59s	remaining: 1h 27m 51s
bestTest = 0.8927472234
bestIteration = 1098
Shrink model to first 1099 iterations.
  Best iteration: 1098


Default metric period is 5 because AUC is/are not implemented for GPU


0:	total: 2.32s	remaining: 42m 26s
100:	total: 3m 31s	remaining: 34m 46s
200:	total: 6m 26s	remaining: 28m 46s
300:	total: 9m 16s	remaining: 24m 33s
400:	total: 11m 56s	remaining: 20m 44s
500:	total: 14m 40s	remaining: 17m 29s
600:	total: 17m 15s	remaining: 14m 16s
700:	total: 19m 52s	remaining: 11m 15s
800:	total: 22m 22s	remaining: 8m 17s
900:	total: 25m	remaining: 5m 28s


# need to run above

In [ ]:
# =============================================================================
# FULL CATBOOST PIPELINE — FIXED VERSION
# Training + 5-Fold CV + SHAP + Confusion Matrix + Feature Importance + Ablation
# =============================================================================
import gc
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, recall_score, precision_score, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, precision_recall_curve
)
import warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# FEATURE SET
# =============================================================================
CAT_FEATURES = ['organism', 'antibiotic', 'antibiotic_class']
NUM_FEATURES = [
    'year', 'log_gdp_per_capita', 'health_expenditure_pct_gdp',
    'population_density_per_sq_km', 'log_consumption_ddd',
    'log_consumption_did_per_class', 'sanitation_access_pct',
    'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius',
    'aware_access_pct'
]
FEATURES = CAT_FEATURES + NUM_FEATURES

print("Categorical features :", CAT_FEATURES)
print("Numerical features   :", NUM_FEATURES)
print("Total features       :", len(FEATURES))

# =============================================================================
# LOAD DATA (Robust)
# =============================================================================
print("\nLoading enriched dataset...")

possible_paths = [
    '/kaggle/working/full_amr_with_macro.csv',
    '/kaggle/input/datasets/david0913/finaldataset/full_amr_with_macro.csv',
    '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv',
    '/kaggle/input/datasets/david0913/full-df/full_amr_with_macro.csv'
]

full_df = None
for path in possible_paths:
    if os.path.exists(path):
        full_df = pd.read_csv(path, low_memory=False)
        print(f"✅ Loaded: {path}")
        break

if full_df is None:
    raise FileNotFoundError("full_amr_with_macro.csv not found!")

full_df = full_df.loc[:, ~full_df.columns.duplicated()]

EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']
source_df = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
del full_df
gc.collect()

# Feature validation
missing = [f for f in FEATURES if f not in source_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
print("✅ All features present.")

# =============================================================================
# PREPARE DATA
# =============================================================================
y_source = source_df['resistance'].values
SCALE_POS_WEIGHT = (y_source == 0).sum() / (y_source == 1).sum()
print(f"\nSource rows      : {len(source_df):,}")
print(f"SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT:.4f}")

TEMPORAL_CUTOFF = 2020
ES_VAL_YEARS = [2019, 2020]

es_val_df   = source_df[source_df['year'].isin(ES_VAL_YEARS)].copy()
es_train_df = source_df[(source_df['year'] <= TEMPORAL_CUTOFF) & 
                       (~source_df['year'].isin(ES_VAL_YEARS))].copy()
test_df     = source_df[source_df['year'] > TEMPORAL_CUTOFF].copy()

print(f"ES train : {len(es_train_df):,}")
print(f"ES val   : {len(es_val_df):,}")
print(f"Test     : {len(test_df):,}")

# =============================================================================
# POOL HELPER
# =============================================================================
def make_pool(df, has_labels=True):
    X = df[FEATURES].copy()
    for col in CAT_FEATURES:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=CAT_FEATURES)
    return Pool(data=X, cat_features=CAT_FEATURES)

# =============================================================================
# TRAINING
# =============================================================================
BASE_PARAMS = {
    'learning_rate': 0.03,
    'depth': 7,
    'l2_leaf_reg': 3,
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 200,
    'early_stopping_rounds': 100,
    'iterations': 5000,
    'task_type': 'GPU',
    'scale_pos_weight': SCALE_POS_WEIGHT,
}

print("\n" + "="*80)
print("Stage 1: Finding best iterations...")
cal_model = CatBoostClassifier(**BASE_PARAMS)
cal_model.fit(make_pool(es_train_df), eval_set=make_pool(es_val_df))
FINAL_ROUNDS = cal_model.best_iteration_
print(f"Best iterations: {FINAL_ROUNDS}")
del cal_model, es_train_df, es_val_df
gc.collect()

print("\nStage 2: Final Model Training...")
final_train_df = source_df[source_df['year'] <= TEMPORAL_CUTOFF].copy()
final_params = {**BASE_PARAMS, 'iterations': FINAL_ROUNDS, 
                'early_stopping_rounds': None, 'verbose': 100}

model_final = CatBoostClassifier(**final_params)
model_final.fit(make_pool(final_train_df))
model_final.save_model(f'{OUTPUT_DIR}/catboost_final.cbm')
print("✅ Model saved: catboost_final.cbm")
del final_train_df
gc.collect()

# =============================================================================
# TEST EVALUATION
# =============================================================================
print("\n" + "="*80)
print("Test Set Performance")
y_te = test_df['resistance'].values
y_prob = model_final.predict_proba(make_pool(test_df))[:, 1]

def compute_metrics(y_true, y_prob, threshold=0.5):
    y_bin = (y_prob >= threshold).astype(int)
    bs = brier_score_loss(y_true, y_prob)
    prev = y_true.mean()
    bss = 1.0 - bs / (prev * (1 - prev)) if prev > 0 else np.nan
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'AUPRC': average_precision_score(y_true, y_prob),
        'F1': f1_score(y_true, y_bin, zero_division=0),
        'Precision': precision_score(y_true, y_bin, zero_division=0),
        'Recall': recall_score(y_true, y_bin, zero_division=0),
        'Brier': bs,
        'BSS': bss,
    }

metrics = compute_metrics(y_te, y_prob)
for k, v in metrics.items():
    print(f"{k:<12}: {v:.4f}")

# =============================================================================
# CONFUSION MATRIX + CURVES
# =============================================================================
THRESHOLD = 0.5
y_pred = (y_prob >= THRESHOLD).astype(int)
cm = confusion_matrix(y_te, y_pred)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay(cm).plot(ax=axes[0], cmap='Blues')
axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_te, y_prob)
axes[1].plot(fpr, tpr, label=f'AUC = {metrics["AUC"]:.4f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve')
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_te, y_prob)
axes[2].plot(rec, prec, label=f'AUPRC = {metrics["AUPRC"]:.4f}')
axes[2].axhline(y_te.mean(), color='gray', linestyle='--')
axes[2].set_title('Precision-Recall Curve')
axes[2].legend()

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/evaluation_plots.png', dpi=200, bbox_inches='tight')
plt.show()

# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================
fi = pd.DataFrame({
    'feature': FEATURES,
    'importance': model_final.get_feature_importance()
}).sort_values('importance', ascending=False)

fi.to_csv(f'{OUTPUT_DIR}/feature_importance.csv', index=False)
print("\nTop 10 Features:")
print(fi.head(10)[['feature','importance']])

# =============================================================================
# SHAP ANALYSIS
# =============================================================================
print("\nComputing SHAP values...")
shap_sample = test_df.sample(min(5000, len(test_df)), random_state=42)
explainer = shap.TreeExplainer(model_final)
shap_values = explainer.shap_values(make_pool(shap_sample, has_labels=False))

shap.summary_plot(shap_values, shap_sample[FEATURES], max_display=15, show=False)
plt.title("SHAP Summary Plot")
plt.savefig(f'{OUTPUT_DIR}/shap_summary.png', dpi=200, bbox_inches='tight')
plt.show()

# =============================================================================
# 5-FOLD TIME-AWARE CV
# =============================================================================
print("\n5-Fold Time-Aware Cross Validation...")
CV_FOLDS = [
    (2015, 2016), (2016, 2017), (2017, 2018),
    (2018, 2019), (2019, 2020)
]
cv_results = []

for train_end, val_year in CV_FOLDS:
    fold_train = source_df[source_df['year'] <= train_end].copy()
    fold_val = source_df[source_df['year'] == val_year].copy()
    
    fold_params = {**BASE_PARAMS, 'iterations': FINAL_ROUNDS, 
                   'early_stopping_rounds': None, 'verbose': 0}
    fold_model = CatBoostClassifier(**fold_params)
    fold_model.fit(make_pool(fold_train))
    
    y_prob_fold = fold_model.predict_proba(make_pool(fold_val))[:, 1]
    fold_metrics = compute_metrics(fold_val['resistance'].values, y_prob_fold)
    cv_results.append({**fold_metrics, 'train_end': train_end, 'val_year': val_year})
    
    del fold_model
    gc.collect()

cv_df = pd.DataFrame(cv_results)
print(cv_df[['train_end', 'val_year', 'AUC', 'AUPRC', 'F1']].round(4))

# =============================================================================
# FINAL MESSAGE
# =============================================================================
print("\n" + "="*80)
print("✅ FULL PIPELINE COMPLETED SUCCESSFULLY!")
print(f"Model saved: {OUTPUT_DIR}/catboost_final.cbm")
print("All plots and tables saved in /kaggle/working/")

# need to run below

In [7]:
import gc
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    f1_score, recall_score, precision_score
)

warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================================================================
# FEATURE SET
# =============================================================================
CAT_FEATURES = ['organism', 'antibiotic', 'antibiotic_class']
NUM_FEATURES = [
    'year', 'log_gdp_per_capita', 'health_expenditure_pct_gdp',
    'population_density_per_sq_km', 'log_consumption_ddd',
    'log_consumption_did_per_class', 'sanitation_access_pct',
    'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius',
    'aware_access_pct'
]
FEATURES = CAT_FEATURES + NUM_FEATURES

print("Categorical features :", CAT_FEATURES)
print("Numerical features   :", NUM_FEATURES)
print("Total features       :", len(FEATURES))

# =============================================================================
# HELPER FUNCTIONS
# =============================================================================
def compute_metrics(y_true, y_prob, threshold=0.5):
    """Compute main evaluation metrics."""
    y_bin = (y_prob >= threshold).astype(int)
    return {
        'AUC':       roc_auc_score(y_true, y_prob),
        'AUPRC':     average_precision_score(y_true, y_prob),
        'F1':        f1_score(y_true, y_bin, zero_division=0),
        'Precision': precision_score(y_true, y_bin, zero_division=0),
        'Recall':    recall_score(y_true, y_bin, zero_division=0),
        'Brier':     brier_score_loss(y_true, y_prob),
    }


def make_pool(df, features=None, has_labels=True):
    """Create CatBoost Pool with proper categorical handling."""
    if features is None:
        features = FEATURES
    cat_feats = [f for f in CAT_FEATURES if f in features]
    X = df[features].copy()
    for col in cat_feats:
        X[col] = X[col].fillna('unknown').astype(str)
    if has_labels:
        return Pool(data=X, label=df['resistance'].values, cat_features=cat_feats)
    return Pool(data=X, cat_features=cat_feats)



print("\nLoading enriched dataset...")
possible_paths = [
    '/kaggle/working/full_amr_with_macro.csv',
    '/kaggle/input/datasets/david0913/finaldataset/full_amr_with_macro.csv',
    '/kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv',
    '/kaggle/input/datasets/david0913/full-df/full_amr_with_macro.csv'
]

full_df = None
for path in possible_paths:
    if os.path.exists(path):
        full_df = pd.read_csv(path, low_memory=False)
        print(f"✅ Loaded: {path}")
        break

if full_df is None:
    raise FileNotFoundError("full_amr_with_macro.csv not found!")

full_df = full_df.loc[:, ~full_df.columns.duplicated()]

EXCLUDE_COUNTRIES = ['Kazakhstan', 'Russia', 'Belarus', 'Ukraine']
source_df = full_df[~full_df['country'].isin(EXCLUDE_COUNTRIES)].copy()
del full_df
gc.collect()

missing = [f for f in FEATURES if f not in source_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

print("✅ All features present.")
print(f"Source rows : {len(source_df):,}")

# =============================================================================
# DATA SPLITS
# =============================================================================
y_source = source_df['resistance'].values
SCALE_POS_WEIGHT = (y_source == 0).sum() / (y_source == 1).sum()
print(f"SCALE_POS_WEIGHT : {SCALE_POS_WEIGHT:.4f}")

TEMPORAL_CUTOFF = 2020
ES_VAL_YEARS    = [2019, 2020]

# Early stopping split (Stage 1 only)
es_val_df   = source_df[source_df['year'].isin(ES_VAL_YEARS)].copy()
es_train_df = source_df[
    (source_df['year'] <= TEMPORAL_CUTOFF) &
    (~source_df['year'].isin(ES_VAL_YEARS))
].copy()

# Held-out test set (post-cutoff)
test_df = source_df[source_df['year'] > TEMPORAL_CUTOFF].copy()

print(f"ES train : {len(es_train_df):,}")
print(f"ES val   : {len(es_val_df):,}")
print(f"Test     : {len(test_df):,}  years={sorted(test_df['year'].unique())}")

if len(test_df) == 0:
    print("⚠️  WARNING: test_df is empty — no data after TEMPORAL_CUTOFF.")

# =============================================================================
# BASE MODEL PARAMS
# =============================================================================
BASE_PARAMS = {
    'learning_rate':        0.03,
    'depth':                7,
    'l2_leaf_reg':          3,
    'eval_metric':          'AUC',
    'random_seed':          42,
    'verbose':              0,
    'early_stopping_rounds': 100,
    'iterations':           5000,
    'task_type':            'GPU',
    'scale_pos_weight':     SCALE_POS_WEIGHT,
}

# =============================================================================
# STAGE 1: Find best iterations via Early Stopping
# =============================================================================
print("\n" + "="*80)
print("Stage 1: Finding optimal iterations with Early Stopping...")

cal_model = CatBoostClassifier(**BASE_PARAMS)
cal_model.fit(make_pool(es_train_df), eval_set=make_pool(es_val_df))

FINAL_ROUNDS = cal_model.best_iteration_
print(f"✅ Best iterations: {FINAL_ROUNDS}")

del cal_model, es_train_df, es_val_df
gc.collect()

# =============================================================================
# STAGE 2: Final Model Training (all data ≤ 2020)
# =============================================================================
print("\nStage 2: Training Final Model on all data ≤ TEMPORAL_CUTOFF...")
final_train_df = source_df[source_df['year'] <= TEMPORAL_CUTOFF].copy()

final_params = {**BASE_PARAMS, 'iterations': FINAL_ROUNDS, 'early_stopping_rounds': None}
model_final = CatBoostClassifier(**final_params)
model_final.fit(make_pool(final_train_df))
model_final.save_model(f'{OUTPUT_DIR}/catboost_final.cbm')
print("✅ Final model saved: catboost_final.cbm")

# Evaluate on test set if available
if len(test_df) > 0:
    y_prob_test = model_final.predict_proba(make_pool(test_df))[:, 1]
    test_metrics = compute_metrics(test_df['resistance'].values, y_prob_test)
    print(f"\nTest Set Metrics:")
    for k, v in test_metrics.items():
        print(f"  {k:10s}: {v:.4f}")

del final_train_df
gc.collect()

# =============================================================================
# STAGE 3: 5-Fold Time-Aware CV with 3-way split (train / es / eval)
#
# Each fold uses three disjoint years:
#   train  → model training
#   es     → early stopping only  (1 year buffer, never evaluated)
#   eval   → honest metric reporting (never seen during training)
#
# This avoids the optimistic bias from using the same set for ES + evaluation.
# Cost: one year of training data per fold. Feasible for folds 3–5.
# Folds 1–2 share es/eval years due to limited early history.
# =============================================================================
print("\n" + "="*80)
print("Stage 3: 5-Fold Time-Aware CV (3-way split per fold)...")

# Format: (train_end, es_year, eval_year)
CV_FOLDS = [
    (2015, 2016, 2017),
    (2016, 2017, 2018),
    (2017, 2018, 2019),
    (2018, 2019, 2020),
    (2019, 2020, 2021),   # eval_year may be empty if data ends at 2020
]

cv_results = []

for train_end, es_year, eval_year in CV_FOLDS:
    fold_train = source_df[source_df['year'] <= train_end].copy()
    fold_es    = source_df[source_df['year'] == es_year].copy()
    fold_eval  = source_df[source_df['year'] == eval_year].copy()

    if len(fold_es) == 0 or len(fold_eval) == 0:
        print(f"  ⚠️  Skipping fold (train≤{train_end}, es={es_year}, eval={eval_year}): insufficient data")
        continue

    print(f"\n→ Fold: train≤{train_end} | ES={es_year} ({len(fold_es):,}) | eval={eval_year} ({len(fold_eval):,})")

    fold_params = {
        **BASE_PARAMS,
        'iterations':            3000,
        'early_stopping_rounds': 80,
    }
    fold_model = CatBoostClassifier(**fold_params)
    # ES uses fold_es — completely separate from fold_eval
    fold_model.fit(make_pool(fold_train), eval_set=make_pool(fold_es))

    # Metrics reported on fold_eval — never touched during training
    y_prob_fold = fold_model.predict_proba(make_pool(fold_eval))[:, 1]
    fold_metrics = compute_metrics(fold_eval['resistance'].values, y_prob_fold)

    cv_results.append({
        **fold_metrics,
        'train_end': train_end,
        'es_year':   es_year,
        'eval_year': eval_year,
        'best_iter': fold_model.best_iteration_,
    })

    del fold_model, fold_train, fold_es, fold_eval
    gc.collect()

cv_df = pd.DataFrame(cv_results)
print("\nCross-Validation Results:")
print(cv_df[['train_end', 'es_year', 'eval_year', 'AUC', 'AUPRC', 'F1', 'Brier', 'best_iter']].round(4))
print(f"\nMean CV AUC   : {cv_df['AUC'].mean():.4f} ± {cv_df['AUC'].std():.4f}")
print(f"Mean CV AUPRC : {cv_df['AUPRC'].mean():.4f} ± {cv_df['AUPRC'].std():.4f}")

cv_df.to_csv(f'{OUTPUT_DIR}/cv_results.csv', index=False)
print("✅ CV results saved: cv_results.csv")

# =============================================================================
# STAGE 4: Feature Group Ablation Study
#
# Removes one feature group at a time and measures AUC on the test set.
# Answers: "which macro features actually drive zero-shot generalization?"
# Critical for paper reviewers evaluating the Kazakhstan prediction.
# =============================================================================
print("\n" + "="*80)
print("Stage 4: Feature Group Ablation Study...")

if len(test_df) == 0:
    print("⚠️  Skipping ablation — test_df is empty.")
else:
    ABLATION_GROUPS = {
        'full_model':      FEATURES,
        'no_gdp':          [f for f in FEATURES if f != 'log_gdp_per_capita'],
        'no_consumption':  [f for f in FEATURES if 'consumption' not in f],
        'no_health_infra': [f for f in FEATURES if f not in
                            ['sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000']],
        'no_climate':      [f for f in FEATURES if f != 'mean_temp_celsius'],
        'no_aware':        [f for f in FEATURES if f != 'aware_access_pct'],
        'macro_only':      CAT_FEATURES + [
                               'year', 'log_gdp_per_capita',
                               'health_expenditure_pct_gdp',
                               'population_density_per_sq_km'
                           ],
        'cat_year_only':   CAT_FEATURES + ['year'],
    }

    ablation_train_df = source_df[source_df['year'] <= TEMPORAL_CUTOFF].copy()
    ablation_results  = []

    for name, feat_subset in ABLATION_GROUPS.items():
        print(f"\n  Ablation: [{name}]  ({len(feat_subset)} features)")

        abl_params = {**BASE_PARAMS, 'iterations': FINAL_ROUNDS, 'early_stopping_rounds': None}
        abl_model  = CatBoostClassifier(**abl_params)
        abl_model.fit(make_pool(ablation_train_df, features=feat_subset))

        y_prob_abl = abl_model.predict_proba(make_pool(test_df, features=feat_subset))[:, 1]
        abl_metrics = compute_metrics(test_df['resistance'].values, y_prob_abl)

        ablation_results.append({
            'ablation':    name,
            'n_features':  len(feat_subset),
            **abl_metrics
        })
        print(f"    AUC={abl_metrics['AUC']:.4f}  AUPRC={abl_metrics['AUPRC']:.4f}  F1={abl_metrics['F1']:.4f}")

        del abl_model
        gc.collect()

    del ablation_train_df
    gc.collect()

    abl_df = pd.DataFrame(ablation_results).sort_values('AUC', ascending=False)
    print("\nAblation Study Results (sorted by AUC):")
    print(abl_df[['ablation', 'n_features', 'AUC', 'AUPRC', 'F1', 'Brier']].round(4).to_string(index=False))

    abl_df.to_csv(f'{OUTPUT_DIR}/ablation_results.csv', index=False)
    print("✅ Ablation results saved: ablation_results.csv")

    # --- Plot AUC delta from full model ---
    full_auc = abl_df.loc[abl_df['ablation'] == 'full_model', 'AUC'].values[0]
    abl_df['AUC_delta'] = abl_df['AUC'] - full_auc

    fig, ax = plt.subplots(figsize=(9, 5))
    colors = ['steelblue' if d >= 0 else 'tomato' for d in abl_df['AUC_delta']]
    ax.barh(abl_df['ablation'], abl_df['AUC_delta'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('ΔAUC vs Full Model')
    ax.set_title('Feature Group Ablation — AUC Impact')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/ablation_auc_delta.png', dpi=150)
    plt.close()
    print("✅ Ablation plot saved: ablation_auc_delta.png")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*80)
print("FULL PIPELINE COMPLETED SUCCESSFULLY!")
print(f"  Final model  → {OUTPUT_DIR}/catboost_final.cbm")
print(f"  CV results   → {OUTPUT_DIR}/cv_results.csv")
print(f"  Ablation     → {OUTPUT_DIR}/ablation_results.csv")
print(f"  Ablation plot→ {OUTPUT_DIR}/ablation_auc_delta.png")

Categorical features : ['organism', 'antibiotic', 'antibiotic_class']
Numerical features   : ['year', 'log_gdp_per_capita', 'health_expenditure_pct_gdp', 'population_density_per_sq_km', 'log_consumption_ddd', 'log_consumption_did_per_class', 'sanitation_access_pct', 'water_access_pct', 'hospital_beds_per_1000', 'mean_temp_celsius', 'aware_access_pct']
Total features       : 14

Loading enriched dataset...
✅ Loaded: /kaggle/input/datasets/uaisamangeldi/totall/full_amr_with_macro.csv
✅ All features present.
Source rows : 20,090,915
SCALE_POS_WEIGHT : 4.5446
ES train : 11,000,251
ES val   : 3,525,708
Test     : 5,564,956  years=[np.int64(2021), np.int64(2022), np.int64(2023)]

Stage 1: Finding optimal iterations with Early Stopping...


Default metric period is 5 because AUC is/are not implemented for GPU


✅ Best iterations: 1925

Stage 2: Training Final Model on all data ≤ TEMPORAL_CUTOFF...


Default metric period is 5 because AUC is/are not implemented for GPU


✅ Final model saved: catboost_final.cbm

Test Set Metrics:
  AUC       : 0.8766
  AUPRC     : 0.6863
  F1        : 0.5750
  Precision : 0.4548
  Recall    : 0.7816
  Brier     : 0.1403

Stage 3: 5-Fold Time-Aware CV (3-way split per fold)...

→ Fold: train≤2015 | ES=2016 (2,359,784) | eval=2017 (2,020,994)


...

    AUC=0.8745  AUPRC=0.6844  F1=0.5727

  Ablation: [macro_only]  (7 features)


Default metric period is 5 because AUC is/are not implemented for GPU


    AUC=0.8725  AUPRC=0.6782  F1=0.5705

  Ablation: [cat_year_only]  (4 features)


Default metric period is 5 because AUC is/are not implemented for GPU


    AUC=0.8400  AUPRC=0.6249  F1=0.5269

Ablation Study Results (sorted by AUC):
       ablation  n_features    AUC  AUPRC     F1  Brier
     full_model          14 0.8771 0.6865 0.5755 0.1406
 no_consumption          12 0.8768 0.6851 0.5711 0.1416
no_health_infra          11 0.8750 0.6831 0.5721 0.1414
         no_gdp          13 0.8748 0.6843 0.5714 0.1430
     no_climate          13 0.8748 0.6848 0.5739 0.1407
       no_aware          13 0.8745 0.6844 0.5727 0.1415
     macro_only           7 0.8725 0.6782 0.5705 0.1406
  cat_year_only           4 0.8400 0.6249 0.5269 0.1592
✅ Ablation results saved: ablation_results.csv
✅ Ablation plot saved: ablation_auc_delta.png

FULL PIPELINE COMPLETED SUCCESSFULLY!
  Final model  → /kaggle/working/catboost_final.cbm
  CV results   → /kaggle/working/cv_results.csv
  Ablation     → /kaggle/working/ablation_results.csv
  Ablation plot→ /kaggle/working/ablation_auc_delta.png


In [ ]:
The file was truncated. Let me read more of the key sections.Now I have a thorough understanding of the notebook. Here's a focused analysis and revised methodologies:

---

## Methodology Issues & Revisions

### 1. **Temporal Split Logic — Inconsistency**
The notebook defines cutoffs differently across cells, causing silent inconsistency:

- Cell 8: `TRAIN_CUTOFF=2018`, `VAL_CUTOFF=2020`
- Cell 19: Same — but Cell 11 uses `TEMPORAL_CUTOFF=2020`, `ES_VAL_YEARS=[2019,2020]`

**Fix:** Define these once at the top and import everywhere:
```python
# config.py / top cell
TRAIN_CUTOFF = 2018
VAL_CUTOFF   = 2020
TEMPORAL_CUTOFF = VAL_CUTOFF  # alias — one source of truth
ES_VAL_YEARS = [TRAIN_CUTOFF + 1, VAL_CUTOFF]
```

---

### 2. **Data Leakage in Label Encoding**
In Cell 19, `LabelEncoder` is fit on `X_train` then mapped via a dict to val/test — but the `lambda` captures `le_dict` by reference in a loop, which is a classic Python closure bug:

```python
# ❌ Bug — all lambdas share the last le_dict
X_val[col] = X_val[col]...map(lambda s, d=le_dict: d.get(s, -1))
```

The `d=le_dict` default-arg trick *does* work here actually — but unseen categories get `-1`, which CatBoost will misinterpret. **Fix:**

```python
# ✅ Safe — use a dedicated transform that maps unknowns explicitly
def safe_encode(series, le):
    known = set(le.classes_)
    return series.fillna('unknown').astype(str).apply(
        lambda x: le.transform([x])[0] if x in known else len(le.classes_)
    )
```
Note: For CatBoost specifically, you should **skip label encoding entirely** and pass raw strings with `cat_features` — CatBoost handles it natively. The encoding is only needed for XGBoost/LightGBM/RF.

---

### 3. **`parse_sir_to_binary` — Slow Scalar Apply**
This is called row-by-row via `apply`, which is very slow on millions of rows.

```python
# ❌ Slow
df['resistance'] = df['SIR'].apply(parse_sir_to_binary)

# ✅ Fast — vectorized
sir_map = {'R': 1, 'RESISTANT': 1, 'S': 0, 'SUSCEPTIBLE': 0}
df['resistance'] = df['SIR'].str.strip().str.upper().map(sir_map)  # NaN for I/unknown
```

---

### 4. **LOCO CV Loop — Retrains Full Model Per Group Unnecessarily**
Cell 11 retrains the full model for each LOCO group (post_soviet, structural_peers, all_except_kz). The `all_except_kz` group already subsumes the others for measuring importance — you only need 1 full retrain + targeted ablations:

```python
# ❌ 3 full retrains just to see group sensitivity
for group_name, countries in loco_groups.items():
    model = CatBoostClassifier(...).fit(...)

# ✅ Train once on all_except_kz, then evaluate on group subsets
model_full = CatBoostClassifier(...).fit(source_except_kz)
for group_name, countries in loco_groups.items():
    eval_df = source_df[source_df['country'].isin(countries)]
    metrics = compute_metrics(eval_df['resistance'], model_full.predict_proba(...)[:,1])
```
This cuts training time ~3×.

---

### 5. **Missing `compute_metrics` in the Final Cell**
Cell 19's `compute_metrics` is defined but truncated mid-function. The CV loop in the last cell calls it without it being in scope. Make sure it's defined **before** the CV loop and includes all needed metrics (AUC, AUPRC, F1, Brier).

---

### 6. **Bootstrap CI — Redundant Loop Structure**
Cell 20 runs 2000 resamples × 4 metrics × 6 models = ~48,000 iterations, each calling `fn()` separately. Batch by model instead:

```python
# ✅ Vectorized bootstrap — compute all metrics per resample
for name, y_prob in raw_preds_test.items():
    boot_idx = rng.integers(0, n, (N_BOOT, n))  # shape: (2000, n)
    for b, idx in enumerate(boot_idx):
        yt, yp = y_test[idx], y_prob[idx]
        if len(np.unique(yt)) < 2: continue
        for metric, fn in METRIC_FNS.items():
            boot_distributions[metric][name][b] = fn(yt, yp)
```

---

### 7. **`I` (Intermediate) Isolates — Silently Dropped Without Reporting**
Discarding `I` isolates is clinically justified, but the notebook never reports *how many* were dropped. Add a brief audit:

```python
i_count = (sir_col.str.upper() == 'I').sum()
print(f"Discarded {i_count:,} intermediate (I) isolates ({i_count/len(df)*100:.1f}%)")
```
This is important for reproducibility and reviewers.

---

### Summary Table

| # | Issue | Impact | Fix |
|---|-------|--------|-----|
| 1 | Inconsistent cutoffs | Silent train/test boundary drift | Single config cell |
| 2 | Label encoding unseen cats → `-1` | Model corruption for XGB/LGBM | `safe_encode()` or skip for CatBoost |
| 3 | Scalar `apply` for SIR parsing | 10–50× slower on large data | Vectorized `str.map()` |
| 4 | LOCO retrains 3× unnecessarily | ~3× wasted GPU time | Train once, evaluate by group |
| 5 | `compute_metrics` out of scope | Runtime crash | Move definition before first call |
| 6 | Bootstrap loop not batched | Slow; inefficient | Pre-generate index matrix |
| 7 | `I` isolates silently dropped | Reproducibility gap | Add audit print |

Want me to apply any of these as code edits directly to the notebook?